In [ ]:
# Find Faster Care Analytics
# Data inspection and cleaning notebook

import pandas as pd
from pathlib import Path

project_path = Path(r"C:\Projects\Find Faster Care Analytics")
raw_data_path = project_path / "Raw Data"
clean_data_path = project_path / "Clean Data"

clean_data_path.mkdir(exist_ok=True)

files = list(raw_data_path.glob("*"))

print("Raw data files found:")
for file in files:
    print(" |", file.name)

print("\nTotal raw files:", len(files))

In [ ]:
# Cell 2 | Inspect sheet names inside each Excel file

for file in files:
    print("=" * 80)
    print("File:", file.name)
    
    try:
        excel_file = pd.ExcelFile(file)
        print("Sheets found:")
        for sheet in excel_file.sheet_names:
            print(" |", sheet)
    except Exception as e:
        print("Could not read file:")
        print(e)

In [ ]:
# Cell 3 | Preview the main sheets before cleaning

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

main_sheets = {
    "RTT incomplete provider": ("rtt_incomplete_provider_apr_2026.xlsx", "Provider"),
    "RTT hospital trust": ("rtt_hospital_trust_apr_2026.xls", "IncompProv"),
    "A&E provider": ("ae_provider_apr_2026.xls", "Provider Level Data"),
    "A&E system mapping": ("ae_system_mapping_apr_2026.xls", "ICB Mapping")
}

for name, (file_name, sheet_name) in main_sheets.items():
    print("\n" + "=" * 100)
    print(name)
    print("File:", file_name)
    print("Sheet:", sheet_name)
    print("=" * 100)
    
    file_path = raw_data_path / file_name
    
    preview = pd.read_excel(file_path, sheet_name=sheet_name, header=None, nrows=20)
    display(preview)

In [ ]:
# Cell 4 | Load main datasets with correct header rows

rtt_incomplete_provider = pd.read_excel(
    raw_data_path / "rtt_incomplete_provider_apr_2026.xlsx",
    sheet_name="Provider",
    header=13
)

rtt_hospital_trust = pd.read_excel(
    raw_data_path / "rtt_hospital_trust_apr_2026.xls",
    sheet_name="IncompProv",
    header=0
)

ae_provider = pd.read_excel(
    raw_data_path / "ae_provider_apr_2026.xls",
    sheet_name="Provider Level Data",
    header=15
)

ae_mapping = pd.read_excel(
    raw_data_path / "ae_system_mapping_apr_2026.xls",
    sheet_name="ICB Mapping",
    header=7
)

datasets = {
    "RTT incomplete provider": rtt_incomplete_provider,
    "RTT hospital trust": rtt_hospital_trust,
    "A&E provider": ae_provider,
    "A&E mapping": ae_mapping
}

for name, df in datasets.items():
    print("=" * 80)
    print(name)
    print("Rows:", df.shape[0])
    print("Columns:", df.shape[1])
    print("First 10 columns:")
    print(list(df.columns[:10]))
    print()

In [ ]:
# Cell 5 | Clean column names and remove blank unnamed columns

import re

def make_column_names_clean(df):
    df = df.copy()
    
    # Drop fully blank unnamed columns
    unnamed_cols = [col for col in df.columns if str(col).lower().startswith("unnamed")]
    for col in unnamed_cols:
        if df[col].isna().all():
            df = df.drop(columns=[col])
    
    clean_columns = []
    
    for col in df.columns:
        col = str(col).strip().lower()
        col = col.replace("%", "percent")
        col = col.replace("&", "and")
        col = col.replace("+", "plus")
        col = col.replace("<", "less_than_")
        col = col.replace(">", "greater_than_")
        col = re.sub(r"[^a-z0-9]+", "_", col)
        col = re.sub(r"_+", "_", col)
        col = col.strip("_")
        clean_columns.append(col)
    
    # Make duplicate column names unique
    seen = {}
    final_columns = []
    
    for col in clean_columns:
        if col not in seen:
            seen[col] = 0
            final_columns.append(col)
        else:
            seen[col] += 1
            final_columns.append(f"{col}_{seen[col]}")
    
    df.columns = final_columns
    return df

# Apply cleaning
rtt_incomplete_provider_clean = make_column_names_clean(rtt_incomplete_provider)
rtt_hospital_trust_clean = make_column_names_clean(rtt_hospital_trust)
ae_provider_clean = make_column_names_clean(ae_provider)
ae_mapping_clean = make_column_names_clean(ae_mapping)

cleaned_datasets = {
    "RTT incomplete provider clean": rtt_incomplete_provider_clean,
    "RTT hospital trust clean": rtt_hospital_trust_clean,
    "A&E provider clean": ae_provider_clean,
    "A&E mapping clean": ae_mapping_clean
}

for name, df in cleaned_datasets.items():
    print("=" * 80)
    print(name)
    print("Rows:", df.shape[0])
    print("Columns:", df.shape[1])
    print("First 12 columns:")
    print(list(df.columns[:12]))
    print()

In [ ]:
# Cell 6 | Basic data quality checks

def data_quality_summary(df, name):
    print("=" * 80)
    print(name)
    print("=" * 80)
    
    print("Rows:", df.shape[0])
    print("Columns:", df.shape[1])
    print("Duplicate rows:", df.duplicated().sum())
    
    missing_summary = df.isna().sum()
    missing_summary = missing_summary[missing_summary > 0].sort_values(ascending=False)
    
    print("\nColumns with missing values:")
    if len(missing_summary) == 0:
        print("No missing values")
    else:
        print(missing_summary.head(15))
    
    print("\nData types:")
    print(df.dtypes.value_counts())
    print("\n")

for name, df in cleaned_datasets.items():
    data_quality_summary(df, name)

In [ ]:
# Cell 7 | Inspect important columns before cleaning

def show_relevant_columns(df, name, keywords):
    print("=" * 80)
    print(name)
    print("=" * 80)
    
    matched_columns = []
    
    for col in df.columns:
        for keyword in keywords:
            if keyword in col:
                matched_columns.append(col)
                break
    
    print("Relevant columns found:")
    for col in matched_columns:
        print(" |", col)
    
    print("\nSample rows:")
    display(df[matched_columns[:12]].head(10))
    print("\n")

keywords = [
    "provider",
    "code",
    "name",
    "treatment",
    "total",
    "percent",
    "wait",
    "attend",
    "admission",
    "icb",
    "region"
]

show_relevant_columns(rtt_incomplete_provider_clean, "RTT incomplete provider", keywords)
show_relevant_columns(ae_provider_clean, "A&E provider", keywords)
show_relevant_columns(ae_mapping_clean, "A&E mapping", keywords)

In [ ]:
# Cell 8 | Create clean RTT provider waiting times table

def find_column(df, possible_names):
    for name in possible_names:
        if name in df.columns:
            return name
    return None

# Find important RTT columns
rtt_columns = {
    "region_code": find_column(rtt_incomplete_provider_clean, ["region_code"]),
    "provider_code": find_column(rtt_incomplete_provider_clean, ["provider_code"]),
    "provider_name": find_column(rtt_incomplete_provider_clean, ["provider_name"]),
    "treatment_function_code": find_column(rtt_incomplete_provider_clean, ["treatment_function_code"]),
    "treatment_function": find_column(rtt_incomplete_provider_clean, ["treatment_function"]),
    "total_incomplete_pathways": find_column(rtt_incomplete_provider_clean, ["total_number_of_incomplete_pathways"]),
    "within_18_weeks": find_column(rtt_incomplete_provider_clean, ["total_within_18_weeks"]),
    "percent_within_18_weeks": find_column(rtt_incomplete_provider_clean, ["percent_within_18_weeks"]),
    "median_wait_weeks": find_column(rtt_incomplete_provider_clean, ["average_median_waiting_time_in_weeks"]),
    "percentile_92_wait_weeks": find_column(rtt_incomplete_provider_clean, ["92nd_percentile_waiting_time_in_weeks"]),
    "wait_52_plus_weeks": find_column(rtt_incomplete_provider_clean, ["total_52_plus_weeks"]),
    "wait_65_plus_weeks": find_column(rtt_incomplete_provider_clean, ["total_65_plus_weeks"]),
    "wait_78_plus_weeks": find_column(rtt_incomplete_provider_clean, ["total_78_plus_weeks"])
}

print("Columns selected:")
for new_name, old_name in rtt_columns.items():
    print(new_name, "=", old_name)

# Keep only columns that exist
available_rtt_columns = {new: old for new, old in rtt_columns.items() if old is not None}

rtt_provider_waiting_times = rtt_incomplete_provider_clean[
    list(available_rtt_columns.values())
].copy()

# Rename columns
rtt_provider_waiting_times = rtt_provider_waiting_times.rename(
    columns={old: new for new, old in available_rtt_columns.items()}
)

# Add reporting period
rtt_provider_waiting_times["reporting_period"] = "2026-04"
rtt_provider_waiting_times["data_source"] = "NHS England RTT Incomplete Provider"

# Convert numeric columns
numeric_columns = [
    "total_incomplete_pathways",
    "within_18_weeks",
    "percent_within_18_weeks",
    "median_wait_weeks",
    "percentile_92_wait_weeks",
    "wait_52_plus_weeks",
    "wait_65_plus_weeks",
    "wait_78_plus_weeks"
]

for col in numeric_columns:
    if col in rtt_provider_waiting_times.columns:
        rtt_provider_waiting_times[col] = pd.to_numeric(
            rtt_provider_waiting_times[col], errors="coerce"
        )

# Remove rows without provider code or treatment function
rtt_provider_waiting_times = rtt_provider_waiting_times.dropna(
    subset=["provider_code", "treatment_function"]
)

# Preview
print("\nClean RTT provider waiting times table")
print("Rows:", rtt_provider_waiting_times.shape[0])
print("Columns:", rtt_provider_waiting_times.shape[1])

display(rtt_provider_waiting_times.head(10))

In [ ]:
# Cell 9 | Create clean A&E provider performance table

def find_exact_or_contains(df, exact_names=None, contains_all=None):
    exact_names = exact_names or []
    contains_all = contains_all or []
    
    # Try exact names first
    for name in exact_names:
        if name in df.columns:
            return name
    
    # Then try columns containing all keywords
    for col in df.columns:
        if all(keyword in col for keyword in contains_all):
            return col
    
    return None

# Find important A&E columns
ae_columns = {
    "provider_code": find_exact_or_contains(ae_provider_clean, exact_names=["code"]),
    "region": find_exact_or_contains(ae_provider_clean, exact_names=["region"]),
    "provider_name": find_exact_or_contains(ae_provider_clean, exact_names=["name"]),
    "total_attendances": find_exact_or_contains(ae_provider_clean, exact_names=["total_attendances"]),
    "attendances_under_4_hours": find_exact_or_contains(ae_provider_clean, exact_names=["total_attendances_less_than_4_hours"]),
    "attendances_over_4_hours": find_exact_or_contains(ae_provider_clean, exact_names=["total_attendances_greater_than_4_hours"]),
    "percent_under_4_hours": find_exact_or_contains(ae_provider_clean, contains_all=["percentage", "4_hours", "all"]),
    "total_emergency_admissions": find_exact_or_contains(ae_provider_clean, exact_names=["total_emergency_admissions"]),
    "patients_waiting_over_4_hours_after_decision_to_admit": find_exact_or_contains(ae_provider_clean, contains_all=["patients", "greater_than_4_hours"]),
    "patients_waiting_over_12_hours_after_decision_to_admit": find_exact_or_contains(ae_provider_clean, contains_all=["patients", "greater_than_12_hours"])
}

print("Columns selected:")
for new_name, old_name in ae_columns.items():
    print(new_name, "=", old_name)

# Keep only columns that exist
available_ae_columns = {new: old for new, old in ae_columns.items() if old is not None}

ae_provider_performance = ae_provider_clean[
    list(available_ae_columns.values())
].copy()

# Rename columns
ae_provider_performance = ae_provider_performance.rename(
    columns={old: new for new, old in available_ae_columns.items()}
)

# Add reporting period
ae_provider_performance["reporting_period"] = "2026-04"
ae_provider_performance["data_source"] = "NHS England A&E Provider"

# Convert numeric columns
ae_numeric_columns = [
    "total_attendances",
    "attendances_under_4_hours",
    "attendances_over_4_hours",
    "percent_under_4_hours",
    "total_emergency_admissions",
    "patients_waiting_over_4_hours_after_decision_to_admit",
    "patients_waiting_over_12_hours_after_decision_to_admit"
]

for col in ae_numeric_columns:
    if col in ae_provider_performance.columns:
        ae_provider_performance[col] = pd.to_numeric(
            ae_provider_performance[col], errors="coerce"
        )

# Remove national summary or blank rows
ae_provider_performance = ae_provider_performance.dropna(
    subset=["provider_code", "provider_name"]
)

# Remove duplicate rows
ae_provider_performance = ae_provider_performance.drop_duplicates()

# Preview
print("\nClean A&E provider performance table")
print("Rows:", ae_provider_performance.shape[0])
print("Columns:", ae_provider_performance.shape[1])

display(ae_provider_performance.head(10))

In [ ]:
# Cell 10 | Create provider reference table

def first_non_null(series):
    values = series.dropna().astype(str).str.strip()
    values = values[values != ""]
    if len(values) == 0:
        return None
    return values.iloc[0]

# RTT provider lookup
rtt_provider_lookup = (
    rtt_provider_waiting_times
    .groupby("provider_code", as_index=False)
    .agg(rtt_provider_name=("provider_name", first_non_null))
)

# A&E provider lookup
ae_provider_lookup = (
    ae_provider_performance
    .groupby("provider_code", as_index=False)
    .agg(
        ae_provider_name=("provider_name", first_non_null),
        ae_region=("region", first_non_null)
    )
)

# A&E mapping lookup
ae_mapping_lookup = ae_mapping_clean.copy()

ae_mapping_lookup = ae_mapping_lookup.rename(columns={
    "code": "provider_code",
    "name": "mapping_provider_name"
})

# Keep useful mapping columns
mapping_columns = [
    "provider_code",
    "mapping_provider_name",
    "icb_code",
    "icb_name",
    "region_name"
]

ae_mapping_lookup = ae_mapping_lookup[
    [col for col in mapping_columns if col in ae_mapping_lookup.columns]
].drop_duplicates()

# Merge all provider references
provider_reference = rtt_provider_lookup.merge(
    ae_provider_lookup,
    on="provider_code",
    how="outer"
)

provider_reference = provider_reference.merge(
    ae_mapping_lookup,
    on="provider_code",
    how="outer"
)

# Create best available provider name
provider_reference["provider_name"] = (
    provider_reference["rtt_provider_name"]
    .combine_first(provider_reference["ae_provider_name"])
    .combine_first(provider_reference["mapping_provider_name"])
)

# Create best available region
provider_reference["region_name"] = (
    provider_reference["region_name"]
    .combine_first(provider_reference["ae_region"])
)

# Final column order
final_provider_columns = [
    "provider_code",
    "provider_name",
    "icb_code",
    "icb_name",
    "region_name",
    "rtt_provider_name",
    "ae_provider_name",
    "mapping_provider_name"
]

provider_reference = provider_reference[
    [col for col in final_provider_columns if col in provider_reference.columns]
]

# Remove blank provider codes
provider_reference = provider_reference.dropna(subset=["provider_code"])

# Remove duplicates
provider_reference = provider_reference.drop_duplicates()

print("Provider reference table")
print("Rows:", provider_reference.shape[0])
print("Columns:", provider_reference.shape[1])

display(provider_reference.head(20))

In [ ]:
# Cell 11 | Create provider pressure summary table

# Aggregate RTT waiting times by provider
rtt_provider_summary = (
    rtt_provider_waiting_times
    .groupby("provider_code", as_index=False)
    .agg(
        total_incomplete_pathways=("total_incomplete_pathways", "sum"),
        total_within_18_weeks=("within_18_weeks", "sum"),
        total_52_plus_weeks=("wait_52_plus_weeks", "sum"),
        total_65_plus_weeks=("wait_65_plus_weeks", "sum"),
        total_78_plus_weeks=("wait_78_plus_weeks", "sum"),
        average_median_wait_weeks=("median_wait_weeks", "mean"),
        average_92nd_percentile_wait_weeks=("percentile_92_wait_weeks", "mean"),
        treatment_functions_count=("treatment_function", "nunique")
    )
)

# Calculate RTT performance
rtt_provider_summary["percent_within_18_weeks_calculated"] = (
    rtt_provider_summary["total_within_18_weeks"] /
    rtt_provider_summary["total_incomplete_pathways"]
)

rtt_provider_summary["percent_52_plus_weeks"] = (
    rtt_provider_summary["total_52_plus_weeks"] /
    rtt_provider_summary["total_incomplete_pathways"]
)

# Aggregate A&E by provider
ae_provider_summary = (
    ae_provider_performance
    .groupby("provider_code", as_index=False)
    .agg(
        total_ae_attendances=("total_attendances", "sum"),
        attendances_under_4_hours=("attendances_under_4_hours", "sum"),
        attendances_over_4_hours=("attendances_over_4_hours", "sum"),
        total_emergency_admissions=("total_emergency_admissions", "sum"),
        patients_waiting_over_4_hours_after_decision_to_admit=(
            "patients_waiting_over_4_hours_after_decision_to_admit", "sum"
        ),
        patients_waiting_over_12_hours_after_decision_to_admit=(
            "patients_waiting_over_12_hours_after_decision_to_admit", "sum"
        )
    )
)

# Calculate A&E performance
ae_provider_summary["percent_ae_under_4_hours"] = (
    ae_provider_summary["attendances_under_4_hours"] /
    ae_provider_summary["total_ae_attendances"]
)

ae_provider_summary["percent_ae_over_4_hours"] = (
    ae_provider_summary["attendances_over_4_hours"] /
    ae_provider_summary["total_ae_attendances"]
)

# Merge RTT, A&E and provider reference
provider_pressure_summary = provider_reference.merge(
    rtt_provider_summary,
    on="provider_code",
    how="left"
)

provider_pressure_summary = provider_pressure_summary.merge(
    ae_provider_summary,
    on="provider_code",
    how="left"
)

# Add reporting period
provider_pressure_summary["reporting_period"] = "2026-04"

# Fill numeric blanks with 0
numeric_cols = provider_pressure_summary.select_dtypes(include=["number"]).columns
provider_pressure_summary[numeric_cols] = provider_pressure_summary[numeric_cols].fillna(0)

# Sort by waiting list size
provider_pressure_summary = provider_pressure_summary.sort_values(
    by="total_incomplete_pathways",
    ascending=False
)

print("Provider pressure summary table")
print("Rows:", provider_pressure_summary.shape[0])
print("Columns:", provider_pressure_summary.shape[1])

display(provider_pressure_summary.head(20))

In [ ]:
# Cell 12 | Create specialty pressure summary table

import numpy as np

specialty_pressure_summary = (
    rtt_provider_waiting_times
    .groupby(["treatment_function_code", "treatment_function"], as_index=False)
    .agg(
        total_incomplete_pathways=("total_incomplete_pathways", "sum"),
        total_within_18_weeks=("within_18_weeks", "sum"),
        total_52_plus_weeks=("wait_52_plus_weeks", "sum"),
        total_65_plus_weeks=("wait_65_plus_weeks", "sum"),
        total_78_plus_weeks=("wait_78_plus_weeks", "sum"),
        average_median_wait_weeks=("median_wait_weeks", "mean"),
        average_92nd_percentile_wait_weeks=("percentile_92_wait_weeks", "mean"),
        provider_count=("provider_code", "nunique")
    )
)

# Calculate performance percentages safely
specialty_pressure_summary["percent_within_18_weeks"] = np.where(
    specialty_pressure_summary["total_incomplete_pathways"] > 0,
    specialty_pressure_summary["total_within_18_weeks"] /
    specialty_pressure_summary["total_incomplete_pathways"],
    0
)

specialty_pressure_summary["percent_52_plus_weeks"] = np.where(
    specialty_pressure_summary["total_incomplete_pathways"] > 0,
    specialty_pressure_summary["total_52_plus_weeks"] /
    specialty_pressure_summary["total_incomplete_pathways"],
    0
)

specialty_pressure_summary["percent_65_plus_weeks"] = np.where(
    specialty_pressure_summary["total_incomplete_pathways"] > 0,
    specialty_pressure_summary["total_65_plus_weeks"] /
    specialty_pressure_summary["total_incomplete_pathways"],
    0
)

specialty_pressure_summary["percent_78_plus_weeks"] = np.where(
    specialty_pressure_summary["total_incomplete_pathways"] > 0,
    specialty_pressure_summary["total_78_plus_weeks"] /
    specialty_pressure_summary["total_incomplete_pathways"],
    0
)

# Add reporting period
specialty_pressure_summary["reporting_period"] = "2026-04"

# Sort by total waiting list size
specialty_pressure_summary = specialty_pressure_summary.sort_values(
    by="total_incomplete_pathways",
    ascending=False
)

print("Specialty pressure summary table")
print("Rows:", specialty_pressure_summary.shape[0])
print("Columns:", specialty_pressure_summary.shape[1])

display(specialty_pressure_summary.head(20))

In [ ]:
# Cell 13 | Create regional and ICB pressure summary table

regional_source = provider_pressure_summary.copy()

# Fill missing names for grouping
if "region_name" in regional_source.columns:
    regional_source["region_name"] = regional_source["region_name"].fillna("Unknown region")
else:
    regional_source["region_name"] = "Unknown region"

if "icb_name" in regional_source.columns:
    regional_source["icb_name"] = regional_source["icb_name"].fillna("Unknown ICB")
else:
    regional_source["icb_name"] = "Unknown ICB"

region_icb_pressure_summary = (
    regional_source
    .groupby(["region_name", "icb_name"], as_index=False)
    .agg(
        provider_count=("provider_code", "nunique"),
        total_incomplete_pathways=("total_incomplete_pathways", "sum"),
        total_within_18_weeks=("total_within_18_weeks", "sum"),
        total_52_plus_weeks=("total_52_plus_weeks", "sum"),
        total_65_plus_weeks=("total_65_plus_weeks", "sum"),
        total_78_plus_weeks=("total_78_plus_weeks", "sum"),
        total_ae_attendances=("total_ae_attendances", "sum"),
        attendances_under_4_hours=("attendances_under_4_hours", "sum"),
        attendances_over_4_hours=("attendances_over_4_hours", "sum"),
        total_emergency_admissions=("total_emergency_admissions", "sum"),
        patients_waiting_over_4_hours_after_decision_to_admit=(
            "patients_waiting_over_4_hours_after_decision_to_admit", "sum"
        ),
        patients_waiting_over_12_hours_after_decision_to_admit=(
            "patients_waiting_over_12_hours_after_decision_to_admit", "sum"
        )
    )
)

# Calculate safe percentages
region_icb_pressure_summary["percent_within_18_weeks"] = np.where(
    region_icb_pressure_summary["total_incomplete_pathways"] > 0,
    region_icb_pressure_summary["total_within_18_weeks"] /
    region_icb_pressure_summary["total_incomplete_pathways"],
    0
)

region_icb_pressure_summary["percent_52_plus_weeks"] = np.where(
    region_icb_pressure_summary["total_incomplete_pathways"] > 0,
    region_icb_pressure_summary["total_52_plus_weeks"] /
    region_icb_pressure_summary["total_incomplete_pathways"],
    0
)

region_icb_pressure_summary["percent_ae_under_4_hours"] = np.where(
    region_icb_pressure_summary["total_ae_attendances"] > 0,
    region_icb_pressure_summary["attendances_under_4_hours"] /
    region_icb_pressure_summary["total_ae_attendances"],
    0
)

region_icb_pressure_summary["percent_ae_over_4_hours"] = np.where(
    region_icb_pressure_summary["total_ae_attendances"] > 0,
    region_icb_pressure_summary["attendances_over_4_hours"] /
    region_icb_pressure_summary["total_ae_attendances"],
    0
)

# Add reporting period
region_icb_pressure_summary["reporting_period"] = "2026-04"

# Sort by waiting list pressure
region_icb_pressure_summary = region_icb_pressure_summary.sort_values(
    by="total_incomplete_pathways",
    ascending=False
)

print("Regional and ICB pressure summary table")
print("Rows:", region_icb_pressure_summary.shape[0])
print("Columns:", region_icb_pressure_summary.shape[1])

display(region_icb_pressure_summary.head(20))

In [ ]:
# Cell 14 | Validate final clean tables before export

final_tables = {
    "rtt_provider_waiting_times": rtt_provider_waiting_times,
    "ae_provider_performance": ae_provider_performance,
    "provider_reference": provider_reference,
    "provider_pressure_summary": provider_pressure_summary,
    "specialty_pressure_summary": specialty_pressure_summary,
    "region_icb_pressure_summary": region_icb_pressure_summary
}

print("=" * 100)
print("FINAL TABLE VALIDATION")
print("=" * 100)

for table_name, df in final_tables.items():
    print("\n" + "=" * 80)
    print(table_name)
    print("=" * 80)
    print("Rows:", df.shape[0])
    print("Columns:", df.shape[1])
    print("Duplicate rows:", df.duplicated().sum())
    print("Missing values:", df.isna().sum().sum())

# Validate RTT totals
rtt_detail_total = rtt_provider_waiting_times["total_incomplete_pathways"].sum()
provider_summary_rtt_total = provider_pressure_summary["total_incomplete_pathways"].sum()
specialty_summary_total = specialty_pressure_summary["total_incomplete_pathways"].sum()

print("\n" + "=" * 100)
print("RTT TOTAL CHECK")
print("=" * 100)
print("RTT detail total incomplete pathways:", rtt_detail_total)
print("Provider summary total incomplete pathways:", provider_summary_rtt_total)
print("Specialty summary total incomplete pathways:", specialty_summary_total)

print("\nRTT provider summary match:", round(rtt_detail_total, 2) == round(provider_summary_rtt_total, 2))
print("RTT specialty summary match:", round(rtt_detail_total, 2) == round(specialty_summary_total, 2))

# Validate A&E totals
ae_detail_total = ae_provider_performance["total_attendances"].sum()
provider_summary_ae_total = provider_pressure_summary["total_ae_attendances"].sum()

print("\n" + "=" * 100)
print("A&E TOTAL CHECK")
print("=" * 100)
print("A&E detail total attendances:", ae_detail_total)
print("Provider summary total A&E attendances:", provider_summary_ae_total)
print("A&E provider summary match:", round(ae_detail_total, 2) == round(provider_summary_ae_total, 2))

# Check percentage columns
percentage_checks = [
    ("provider_pressure_summary", provider_pressure_summary, [
        "percent_within_18_weeks_calculated",
        "percent_52_plus_weeks",
        "percent_ae_under_4_hours",
        "percent_ae_over_4_hours"
    ]),
    ("specialty_pressure_summary", specialty_pressure_summary, [
        "percent_within_18_weeks",
        "percent_52_plus_weeks",
        "percent_65_plus_weeks",
        "percent_78_plus_weeks"
    ]),
    ("region_icb_pressure_summary", region_icb_pressure_summary, [
        "percent_within_18_weeks",
        "percent_52_plus_weeks",
        "percent_ae_under_4_hours",
        "percent_ae_over_4_hours"
    ])
]

print("\n" + "=" * 100)
print("PERCENTAGE RANGE CHECK")
print("=" * 100)

for table_name, df, columns in percentage_checks:
    print("\n", table_name)
    for col in columns:
        if col in df.columns:
            invalid_count = ((df[col] < 0) | (df[col] > 1)).sum()
            print(col, "| invalid values:", invalid_count)

In [ ]:
# Cell 15 | Export final clean tables to CSV

clean_data_path.mkdir(exist_ok=True)

# Export clean tables
rtt_provider_waiting_times.to_csv(
    clean_data_path / "rtt_provider_waiting_times.csv",
    index=False,
    encoding="utf-8-sig"
)

ae_provider_performance.to_csv(
    clean_data_path / "ae_provider_performance.csv",
    index=False,
    encoding="utf-8-sig"
)

provider_reference.to_csv(
    clean_data_path / "provider_reference.csv",
    index=False,
    encoding="utf-8-sig"
)

provider_pressure_summary.to_csv(
    clean_data_path / "provider_pressure_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

specialty_pressure_summary.to_csv(
    clean_data_path / "specialty_pressure_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

region_icb_pressure_summary.to_csv(
    clean_data_path / "region_icb_pressure_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

# Create validation summary
validation_summary = []

for table_name, df in final_tables.items():
    validation_summary.append({
        "table_name": table_name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "duplicate_rows": df.duplicated().sum(),
        "missing_values": df.isna().sum().sum()
    })

validation_summary_df = pd.DataFrame(validation_summary)

validation_summary_df.to_csv(
    clean_data_path / "validation_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Clean files exported successfully to:")
print(clean_data_path)

print("\nFiles saved:")
for file in clean_data_path.glob("*.csv"):
    print(" |", file.name)

In [ ]:
!pip install sqlalchemy psycopg2-binary

In [ ]:
# Cell 17 | Connect to PostgreSQL and upload clean CSV files

import pandas as pd
from pathlib import Path
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
import getpass

# Project paths
project_path = Path(r"C:\Projects\Find Faster Care Analytics")
clean_data_path = project_path / "Clean Data"

# PostgreSQL connection details
db_user = "postgres"
db_password = getpass.getpass("Enter your PostgreSQL password: ")
db_host = "localhost"
db_port = "5432"
db_name = "find_faster_care_analytics"

# Create database engine
connection_string = (
    f"postgresql+psycopg2://{db_user}:{quote_plus(db_password)}"
    f"@{db_host}:{db_port}/{db_name}"
)

engine = create_engine(connection_string)

# Test connection
with engine.connect() as connection:
    result = connection.execute(text("SELECT 1"))
    print("Database connection successful:", result.scalar())

# CSV files to upload
csv_tables = {
    "rtt_provider_waiting_times": "rtt_provider_waiting_times.csv",
    "ae_provider_performance": "ae_provider_performance.csv",
    "provider_reference": "provider_reference.csv",
    "provider_pressure_summary": "provider_pressure_summary.csv",
    "specialty_pressure_summary": "specialty_pressure_summary.csv",
    "region_icb_pressure_summary": "region_icb_pressure_summary.csv",
    "validation_summary": "validation_summary.csv"
}

# Upload each CSV to PostgreSQL
for table_name, file_name in csv_tables.items():
    file_path = clean_data_path / file_name
    
    df = pd.read_csv(file_path)
    
    df.to_sql(
        table_name,
        engine,
        if_exists="replace",
        index=False
    )
    
    print(f"Uploaded {table_name}: {df.shape[0]} rows, {df.shape[1]} columns")

print("\nAll clean tables uploaded to PostgreSQL successfully.")

In [ ]:
# Cell 18 | Verify tables inside PostgreSQL

from sqlalchemy import text

table_names = [
    "rtt_provider_waiting_times",
    "ae_provider_performance",
    "provider_reference",
    "provider_pressure_summary",
    "specialty_pressure_summary",
    "region_icb_pressure_summary",
    "validation_summary"
]

with engine.connect() as connection:
    print("PostgreSQL table row counts")
    print("=" * 60)
    
    for table in table_names:
        result = connection.execute(text(f"SELECT COUNT(*) FROM {table}"))
        row_count = result.scalar()
        print(f"{table}: {row_count} rows")

In [ ]:
# Cell 19 | Create SQL analysis views for Power BI

from pathlib import Path
from sqlalchemy import text

project_path = Path(r"C:\Projects\Find Faster Care Analytics")
sql_path = project_path / "SQL"
sql_path.mkdir(exist_ok=True)

sql_script = """
DROP VIEW IF EXISTS powerbi_provider_pressure;
DROP VIEW IF EXISTS powerbi_specialty_pressure;
DROP VIEW IF EXISTS powerbi_region_icb_pressure;
DROP VIEW IF EXISTS powerbi_ae_provider_performance;
DROP VIEW IF EXISTS powerbi_rtt_provider_waiting_times;

CREATE VIEW powerbi_provider_pressure AS
SELECT
    provider_code,
    provider_name,
    icb_code,
    icb_name,
    region_name,
    reporting_period,
    total_incomplete_pathways,
    total_within_18_weeks,
    total_52_plus_weeks,
    total_65_plus_weeks,
    total_78_plus_weeks,
    average_median_wait_weeks,
    average_92nd_percentile_wait_weeks,
    treatment_functions_count,
    percent_within_18_weeks_calculated,
    percent_52_plus_weeks,
    total_ae_attendances,
    attendances_under_4_hours,
    attendances_over_4_hours,
    total_emergency_admissions,
    patients_waiting_over_4_hours_after_decision_to_admit,
    patients_waiting_over_12_hours_after_decision_to_admit,
    percent_ae_under_4_hours,
    percent_ae_over_4_hours,
    CASE
        WHEN total_incomplete_pathways >= 100000 THEN 'Very High Waiting List'
        WHEN total_incomplete_pathways >= 50000 THEN 'High Waiting List'
        WHEN total_incomplete_pathways >= 10000 THEN 'Medium Waiting List'
        ELSE 'Lower Waiting List'
    END AS waiting_list_pressure_group,
    CASE
        WHEN percent_within_18_weeks_calculated >= 0.70 THEN 'Stronger RTT Performance'
        WHEN percent_within_18_weeks_calculated >= 0.60 THEN 'Moderate RTT Performance'
        ELSE 'Lower RTT Performance'
    END AS rtt_performance_group,
    CASE
        WHEN percent_ae_under_4_hours >= 0.76 THEN 'Stronger A&E Performance'
        WHEN percent_ae_under_4_hours >= 0.65 THEN 'Moderate A&E Performance'
        ELSE 'Lower A&E Performance'
    END AS ae_performance_group
FROM provider_pressure_summary;

CREATE VIEW powerbi_specialty_pressure AS
SELECT
    treatment_function_code,
    treatment_function,
    reporting_period,
    total_incomplete_pathways,
    total_within_18_weeks,
    total_52_plus_weeks,
    total_65_plus_weeks,
    total_78_plus_weeks,
    average_median_wait_weeks,
    average_92nd_percentile_wait_weeks,
    provider_count,
    percent_within_18_weeks,
    percent_52_plus_weeks,
    percent_65_plus_weeks,
    percent_78_plus_weeks,
    CASE
        WHEN total_incomplete_pathways >= 500000 THEN 'Very High Specialty Pressure'
        WHEN total_incomplete_pathways >= 250000 THEN 'High Specialty Pressure'
        WHEN total_incomplete_pathways >= 100000 THEN 'Medium Specialty Pressure'
        ELSE 'Lower Specialty Pressure'
    END AS specialty_pressure_group
FROM specialty_pressure_summary;

CREATE VIEW powerbi_region_icb_pressure AS
SELECT
    region_name,
    icb_name,
    reporting_period,
    provider_count,
    total_incomplete_pathways,
    total_within_18_weeks,
    total_52_plus_weeks,
    total_65_plus_weeks,
    total_78_plus_weeks,
    total_ae_attendances,
    attendances_under_4_hours,
    attendances_over_4_hours,
    total_emergency_admissions,
    patients_waiting_over_4_hours_after_decision_to_admit,
    patients_waiting_over_12_hours_after_decision_to_admit,
    percent_within_18_weeks,
    percent_52_plus_weeks,
    percent_ae_under_4_hours,
    percent_ae_over_4_hours
FROM region_icb_pressure_summary;

CREATE VIEW powerbi_ae_provider_performance AS
SELECT
    provider_code,
    provider_name,
    region,
    reporting_period,
    total_attendances,
    attendances_under_4_hours,
    attendances_over_4_hours,
    percent_under_4_hours,
    total_emergency_admissions,
    patients_waiting_over_4_hours_after_decision_to_admit,
    patients_waiting_over_12_hours_after_decision_to_admit,
    data_source
FROM ae_provider_performance;

CREATE VIEW powerbi_rtt_provider_waiting_times AS
SELECT
    provider_code,
    provider_name,
    region_code,
    treatment_function_code,
    treatment_function,
    reporting_period,
    total_incomplete_pathways,
    within_18_weeks,
    percent_within_18_weeks,
    median_wait_weeks,
    percentile_92_wait_weeks,
    wait_52_plus_weeks,
    wait_65_plus_weeks,
    wait_78_plus_weeks,
    data_source
FROM rtt_provider_waiting_times;
"""

# Run SQL script in PostgreSQL
with engine.begin() as connection:
    connection.execute(text(sql_script))

# Save SQL script to SQL folder
sql_file_path = sql_path / "01_create_powerbi_views.sql"

with open(sql_file_path, "w", encoding="utf-8") as file:
    file.write(sql_script)

print("SQL Power BI views created successfully.")
print("SQL script saved to:")
print(sql_file_path)

In [ ]:
# Cell 20 | Verify Power BI SQL views

powerbi_views = [
    "powerbi_provider_pressure",
    "powerbi_specialty_pressure",
    "powerbi_region_icb_pressure",
    "powerbi_ae_provider_performance",
    "powerbi_rtt_provider_waiting_times"
]

with engine.connect() as connection:
    print("Power BI view row counts")
    print("=" * 60)
    
    for view in powerbi_views:
        result = connection.execute(text(f"SELECT COUNT(*) FROM {view}"))
        row_count = result.scalar()
        print(f"{view}: {row_count} rows")

In [ ]:
# Cell 21 | Create final project summary and data dictionary

# Create data dictionary for Power BI views
data_dictionary_rows = []

with engine.connect() as connection:
    for view in powerbi_views:
        query = text("""
            SELECT column_name, data_type
            FROM information_schema.columns
            WHERE table_name = :view_name
            ORDER BY ordinal_position
        """)
        
        result = connection.execute(query, {"view_name": view})
        
        for row in result:
            data_dictionary_rows.append({
                "view_name": view,
                "column_name": row.column_name,
                "data_type": row.data_type
            })

data_dictionary = pd.DataFrame(data_dictionary_rows)

data_dictionary.to_csv(
    clean_data_path / "powerbi_data_dictionary.csv",
    index=False,
    encoding="utf-8-sig"
)

# Final summary
print("=" * 100)
print("FIND FASTER CARE ANALYTICS | DATA PREPARATION COMPLETE")
print("=" * 100)

print("\nProject:")
print("Find Faster Care Analytics | NHS Waiting Times and Provider Performance")

print("\nBusiness problem:")
print("How can published NHS waiting time and urgent care data be used to compare provider pressure, identify access delays and support patient friendly healthcare access insights?")

print("\nClean CSV files created:")
for file in clean_data_path.glob("*.csv"):
    print(" |", file.name)

print("\nPostgreSQL database:")
print(" | find_faster_care_analytics")

print("\nPower BI ready views:")
for view in powerbi_views:
    print(" |", view)

print("\nNext step:")
print("Connect Power BI to PostgreSQL and build the dashboard.")

In [ ]:
import pandas as pd
from pathlib import Path

project_path = Path(r"C:\Projects\Find Faster Care Analytics")
clean_data_path = project_path / "Clean Data"

rtt_file = clean_data_path / "rtt_provider_waiting_times.csv"

print("File exists:", rtt_file.exists())
print("File path:", rtt_file)

rtt = pd.read_csv(rtt_file)

print("Rows:", len(rtt))
print("Columns:", len(rtt.columns))
print(rtt.columns.tolist()[:20])

In [ ]:
# Fix double counting issue
# Split RTT rows into provider total rows and specialty rows

rtt["treatment_function_clean"] = (
    rtt["treatment_function"]
    .astype(str)
    .str.strip()
)

rtt_total_rows = rtt[
    rtt["treatment_function_clean"].str.lower() == "total"
].copy()

rtt_specialty_rows = rtt[
    rtt["treatment_function_clean"].str.lower() != "total"
].copy()

print("RTT all rows total:")
print(rtt["total_incomplete_pathways"].sum())

print("\nRTT provider TOTAL rows only:")
print(rtt_total_rows["total_incomplete_pathways"].sum())

print("\nRTT specialty rows only:")
print(rtt_specialty_rows["total_incomplete_pathways"].sum())

print("\nRows check")
print("All RTT rows:", len(rtt))
print("Total rows:", len(rtt_total_rows))
print("Specialty rows:", len(rtt_specialty_rows))

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import shutil

project_path = Path(r"C:\Projects\Find Faster Care Analytics")
clean_data_path = project_path / "Clean Data"

rtt_path = clean_data_path / "rtt_provider_waiting_times.csv"
provider_pressure_path = clean_data_path / "provider_pressure_summary.csv"
specialty_pressure_path = clean_data_path / "specialty_pressure_summary.csv"
region_icb_pressure_path = clean_data_path / "region_icb_pressure_summary.csv"
validation_summary_path = clean_data_path / "validation_summary.csv"

backup_folder = clean_data_path / "backup_before_double_count_fix"
backup_folder.mkdir(exist_ok=True)

for file_path in [
    provider_pressure_path,
    specialty_pressure_path,
    region_icb_pressure_path,
    validation_summary_path
]:
    if file_path.exists():
        shutil.copy2(file_path, backup_folder / file_path.name)

print("Backups saved to:", backup_folder)

rtt = pd.read_csv(rtt_path)
provider_pressure_old = pd.read_csv(provider_pressure_path)

rtt["treatment_function_clean"] = (
    rtt["treatment_function"]
    .astype(str)
    .str.strip()
)

rtt_total_rows = rtt[
    rtt["treatment_function_clean"].str.lower() == "total"
].copy()

rtt_specialty_rows = rtt[
    rtt["treatment_function_clean"].str.lower() != "total"
].copy()

print("\nDouble count check")
print("All RTT rows total:", rtt["total_incomplete_pathways"].sum())
print("Provider total rows only:", rtt_total_rows["total_incomplete_pathways"].sum())
print("Specialty rows only:", rtt_specialty_rows["total_incomplete_pathways"].sum())

rtt_provider_summary = (
    rtt_total_rows
    .groupby("provider_code", as_index=False)
    .agg(
        total_incomplete_pathways=("total_incomplete_pathways", "sum"),
        total_within_18_weeks=("within_18_weeks", "sum"),
        total_52_plus_weeks=("wait_52_plus_weeks", "sum"),
        total_65_plus_weeks=("wait_65_plus_weeks", "sum"),
        total_78_plus_weeks=("wait_78_plus_weeks", "sum"),
        average_median_wait_weeks=("median_wait_weeks", "mean"),
        average_92nd_percentile_wait_weeks=("percentile_92_wait_weeks", "mean"),
        treatment_functions_count=("treatment_function", "nunique")
    )
)

rtt_provider_summary["percent_within_18_weeks_calculated"] = np.where(
    rtt_provider_summary["total_incomplete_pathways"] > 0,
    rtt_provider_summary["total_within_18_weeks"] / rtt_provider_summary["total_incomplete_pathways"],
    0
)

rtt_provider_summary["percent_52_plus_weeks"] = np.where(
    rtt_provider_summary["total_incomplete_pathways"] > 0,
    rtt_provider_summary["total_52_plus_weeks"] / rtt_provider_summary["total_incomplete_pathways"],
    0
)

provider_pressure = provider_pressure_old.copy()

rtt_cols_to_replace = [
    "total_incomplete_pathways",
    "total_within_18_weeks",
    "total_52_plus_weeks",
    "total_65_plus_weeks",
    "total_78_plus_weeks",
    "average_median_wait_weeks",
    "average_92nd_percentile_wait_weeks",
    "treatment_functions_count",
    "percent_within_18_weeks_calculated",
    "percent_52_plus_weeks"
]

provider_lookup = rtt_provider_summary.set_index("provider_code")

for col in rtt_cols_to_replace:
    provider_pressure[col] = provider_pressure["provider_code"].map(provider_lookup[col])

for col in rtt_cols_to_replace:
    if col in provider_pressure.columns:
        provider_pressure[col] = provider_pressure[col].fillna(0)

specialty_pressure_summary = (
    rtt_specialty_rows
    .groupby(["treatment_function_code", "treatment_function"], as_index=False)
    .agg(
        total_incomplete_pathways=("total_incomplete_pathways", "sum"),
        total_within_18_weeks=("within_18_weeks", "sum"),
        total_52_plus_weeks=("wait_52_plus_weeks", "sum"),
        total_65_plus_weeks=("wait_65_plus_weeks", "sum"),
        total_78_plus_weeks=("wait_78_plus_weeks", "sum"),
        average_median_wait_weeks=("median_wait_weeks", "mean"),
        average_92nd_percentile_wait_weeks=("percentile_92_wait_weeks", "mean"),
        provider_count=("provider_code", "nunique")
    )
)

specialty_pressure_summary["percent_within_18_weeks"] = np.where(
    specialty_pressure_summary["total_incomplete_pathways"] > 0,
    specialty_pressure_summary["total_within_18_weeks"] / specialty_pressure_summary["total_incomplete_pathways"],
    0
)

specialty_pressure_summary["percent_52_plus_weeks"] = np.where(
    specialty_pressure_summary["total_incomplete_pathways"] > 0,
    specialty_pressure_summary["total_52_plus_weeks"] / specialty_pressure_summary["total_incomplete_pathways"],
    0
)

specialty_pressure_summary["percent_65_plus_weeks"] = np.where(
    specialty_pressure_summary["total_incomplete_pathways"] > 0,
    specialty_pressure_summary["total_65_plus_weeks"] / specialty_pressure_summary["total_incomplete_pathways"],
    0
)

specialty_pressure_summary["percent_78_plus_weeks"] = np.where(
    specialty_pressure_summary["total_incomplete_pathways"] > 0,
    specialty_pressure_summary["total_78_plus_weeks"] / specialty_pressure_summary["total_incomplete_pathways"],
    0
)

specialty_pressure_summary["reporting_period"] = "2026-04"

regional_source = provider_pressure.copy()

regional_source["region_name"] = regional_source["region_name"].fillna("Unknown region")
regional_source["icb_name"] = regional_source["icb_name"].fillna("Unknown ICB")

region_icb_pressure_summary = (
    regional_source
    .groupby(["region_name", "icb_name"], as_index=False)
    .agg(
        provider_count=("provider_code", "nunique"),
        total_incomplete_pathways=("total_incomplete_pathways", "sum"),
        total_within_18_weeks=("total_within_18_weeks", "sum"),
        total_52_plus_weeks=("total_52_plus_weeks", "sum"),
        total_65_plus_weeks=("total_65_plus_weeks", "sum"),
        total_78_plus_weeks=("total_78_plus_weeks", "sum"),
        total_ae_attendances=("total_ae_attendances", "sum"),
        attendances_under_4_hours=("attendances_under_4_hours", "sum"),
        attendances_over_4_hours=("attendances_over_4_hours", "sum"),
        total_emergency_admissions=("total_emergency_admissions", "sum"),
        patients_waiting_over_4_hours_after_decision_to_admit=(
            "patients_waiting_over_4_hours_after_decision_to_admit", "sum"
        ),
        patients_waiting_over_12_hours_after_decision_to_admit=(
            "patients_waiting_over_12_hours_after_decision_to_admit", "sum"
        )
    )
)

region_icb_pressure_summary["percent_within_18_weeks"] = np.where(
    region_icb_pressure_summary["total_incomplete_pathways"] > 0,
    region_icb_pressure_summary["total_within_18_weeks"] / region_icb_pressure_summary["total_incomplete_pathways"],
    0
)

region_icb_pressure_summary["percent_52_plus_weeks"] = np.where(
    region_icb_pressure_summary["total_incomplete_pathways"] > 0,
    region_icb_pressure_summary["total_52_plus_weeks"] / region_icb_pressure_summary["total_incomplete_pathways"],
    0
)

region_icb_pressure_summary["percent_ae_under_4_hours"] = np.where(
    region_icb_pressure_summary["total_ae_attendances"] > 0,
    region_icb_pressure_summary["attendances_under_4_hours"] / region_icb_pressure_summary["total_ae_attendances"],
    0
)

region_icb_pressure_summary["percent_ae_over_4_hours"] = np.where(
    region_icb_pressure_summary["total_ae_attendances"] > 0,
    region_icb_pressure_summary["attendances_over_4_hours"] / region_icb_pressure_summary["total_ae_attendances"],
    0
)

provider_pressure.to_csv(provider_pressure_path, index=False, encoding="utf-8-sig")
specialty_pressure_summary.to_csv(specialty_pressure_path, index=False, encoding="utf-8-sig")
region_icb_pressure_summary.to_csv(region_icb_pressure_path, index=False, encoding="utf-8-sig")

validation_summary = pd.DataFrame([
    {
        "table_name": "provider_pressure_summary",
        "rows": provider_pressure.shape[0],
        "columns": provider_pressure.shape[1],
        "duplicate_rows": provider_pressure.duplicated().sum(),
        "missing_values": provider_pressure.isna().sum().sum()
    },
    {
        "table_name": "specialty_pressure_summary",
        "rows": specialty_pressure_summary.shape[0],
        "columns": specialty_pressure_summary.shape[1],
        "duplicate_rows": specialty_pressure_summary.duplicated().sum(),
        "missing_values": specialty_pressure_summary.isna().sum().sum()
    },
    {
        "table_name": "region_icb_pressure_summary",
        "rows": region_icb_pressure_summary.shape[0],
        "columns": region_icb_pressure_summary.shape[1],
        "duplicate_rows": region_icb_pressure_summary.duplicated().sum(),
        "missing_values": region_icb_pressure_summary.isna().sum().sum()
    }
])

validation_summary.to_csv(validation_summary_path, index=False, encoding="utf-8-sig")

print("\nCorrected files saved")
print(provider_pressure_path)
print(specialty_pressure_path)
print(region_icb_pressure_path)
print(validation_summary_path)

print("\nValidation totals")
print("Correct provider total:", provider_pressure["total_incomplete_pathways"].sum())
print("Correct specialty total:", specialty_pressure_summary["total_incomplete_pathways"].sum())
print("Correct region ICB total:", region_icb_pressure_summary["total_incomplete_pathways"].sum())

print("\nRows")
print("Provider pressure rows:", len(provider_pressure))
print("Specialty pressure rows:", len(specialty_pressure_summary))
print("Region ICB rows:", len(region_icb_pressure_summary))

In [ ]:
import psycopg2
from pathlib import Path
from getpass import getpass

project_path = Path(r"C:\Projects\Find Faster Care Analytics")
sql_file_path = project_path / "SQL" / "01_create_powerbi_views.sql"

password = getpass("Enter your PostgreSQL password: ")

conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="find_faster_care_analytics",
    user="postgres",
    password=password
)

conn.autocommit = True
cur = conn.cursor()

# Add missing reporting_period column to corrected summary tables
fix_reporting_period_sql = """
ALTER TABLE provider_pressure_summary
ADD COLUMN IF NOT EXISTS reporting_period TEXT;

UPDATE provider_pressure_summary
SET reporting_period = '2026-04'
WHERE reporting_period IS NULL OR reporting_period = '';

ALTER TABLE specialty_pressure_summary
ADD COLUMN IF NOT EXISTS reporting_period TEXT;

UPDATE specialty_pressure_summary
SET reporting_period = '2026-04'
WHERE reporting_period IS NULL OR reporting_period = '';

ALTER TABLE region_icb_pressure_summary
ADD COLUMN IF NOT EXISTS reporting_period TEXT;

UPDATE region_icb_pressure_summary
SET reporting_period = '2026-04'
WHERE reporting_period IS NULL OR reporting_period = '';
"""

cur.execute(fix_reporting_period_sql)
print("reporting_period fixed in summary tables.")

# Recreate Power BI views
with open(sql_file_path, "r", encoding="utf-8-sig") as f:
    sql_script = f.read()

cur.execute(sql_script)
print("Power BI views recreated successfully.")

# Quick validation
cur.execute("""
SELECT 'powerbi_provider_pressure' AS view_name, COUNT(*) FROM powerbi_provider_pressure
UNION ALL
SELECT 'powerbi_specialty_pressure', COUNT(*) FROM powerbi_specialty_pressure
UNION ALL
SELECT 'powerbi_region_icb_pressure', COUNT(*) FROM powerbi_region_icb_pressure;
""")

for row in cur.fetchall():
    print(row)

cur.close()
conn.close()

In [ ]:
import pandas as pd
from pathlib import Path
import zipfile

project_path = Path(r"C:\Projects\Find Faster Care Analytics")
raw_data_path = project_path / "Raw Data"
clean_data_path = project_path / "Clean Data"

clean_data_path.mkdir(exist_ok=True)

zip_files = list(raw_data_path.glob("*.zip"))

print("Zip files found:")
for file in zip_files:
    print(file.name)

if len(zip_files) == 0:
    print("\nNo zip file found. Put the NHS zip file inside Raw Data folder first.")
else:
    nhs_zip_path = zip_files[0]
    print("\nUsing zip file:")
    print(nhs_zip_path)

    extract_path = raw_data_path / "nhs_full_csv_extract_apr26"
    extract_path.mkdir(exist_ok=True)

    with zipfile.ZipFile(nhs_zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_path)

    csv_files = list(extract_path.glob("*.csv"))

    print("\nCSV files extracted:")
    for file in csv_files:
        print(file.name)

    nhs_rtt_csv = csv_files[0]
    print("\nUsing CSV file:")
    print(nhs_rtt_csv)

In [ ]:
import pandas as pd

nhs_rtt = pd.read_csv(nhs_rtt_csv, low_memory=False)

print("Rows:", nhs_rtt.shape[0])
print("Columns:", nhs_rtt.shape[1])

print("\nFirst 30 columns:")
for col in nhs_rtt.columns[:30]:
    print(col)

print("\nColumns containing RTT or Part:")
for col in nhs_rtt.columns:
    if "RTT" in col or "Part" in col or "Treatment" in col or "Pathway" in col:
        print(col)

print("\nFirst 5 rows:")
display(nhs_rtt.head())

In [ ]:
# Inspect NHS RTT full CSV structure

print("RTT Part Description values:")
print(nhs_rtt["RTT Part Description"].dropna().unique())

print("\nRTT Part Type values:")
print(nhs_rtt["RTT Part Type"].dropna().unique())

print("\nTreatment functions containing Total:")
print(
    nhs_rtt[
        nhs_rtt["Treatment Function Name"].astype(str).str.contains("Total", case=False, na=False)
    ]["Treatment Function Name"].dropna().unique()
)

print("\nColumns containing 52, 65, 78, 92, 18, Total:")
for col in nhs_rtt.columns:
    col_text = str(col).lower()
    if "52" in col_text or "65" in col_text or "78" in col_text or "92" in col_text or "18" in col_text or "total" in col_text:
        print(col)

print("\nColumns containing Weeks:")
week_cols = [col for col in nhs_rtt.columns if "Weeks" in str(col)]
for col in week_cols:
    print(col)

print("\nNumber of week columns:", len(week_cols))

In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
import shutil

project_path = Path(r"C:\Projects\Find Faster Care Analytics")
clean_data_path = project_path / "Clean Data"

rtt_provider_path = clean_data_path / "rtt_provider_waiting_times.csv"
provider_pressure_path = clean_data_path / "provider_pressure_summary.csv"
specialty_pressure_path = clean_data_path / "specialty_pressure_summary.csv"
region_icb_pressure_path = clean_data_path / "region_icb_pressure_summary.csv"
validation_summary_path = clean_data_path / "validation_summary.csv"

backup_folder = clean_data_path / "backup_before_nhs_full_csv_rebuild"
backup_folder.mkdir(exist_ok=True)

for file_path in [
    rtt_provider_path,
    provider_pressure_path,
    specialty_pressure_path,
    region_icb_pressure_path,
    validation_summary_path
]:
    if file_path.exists():
        shutil.copy2(file_path, backup_folder / file_path.name)

print("Backups saved to:", backup_folder)

# Keep old provider file for A&E and mapping columns
old_provider_pressure = pd.read_csv(provider_pressure_path)
old_region_icb = pd.read_csv(region_icb_pressure_path)

# Filter NHS full CSV to incomplete pathways only
rtt_full = nhs_rtt.copy()

rtt_full["RTT Part Description Clean"] = (
    rtt_full["RTT Part Description"]
    .astype(str)
    .str.strip()
)

rtt_full["Treatment Function Name Clean"] = (
    rtt_full["Treatment Function Name"]
    .astype(str)
    .str.strip()
)

incomplete_rows = rtt_full[
    rtt_full["RTT Part Description Clean"].str.lower() == "incomplete pathways"
].copy()

total_rows = incomplete_rows[
    incomplete_rows["Treatment Function Name Clean"].str.lower() == "total"
].copy()

specialty_rows = incomplete_rows[
    incomplete_rows["Treatment Function Name Clean"].str.lower() != "total"
].copy()

# Find week columns
week_cols = []
week_starts = []

for col in incomplete_rows.columns:
    match = re.search(r"Gt\s+(\d+)(?:\s+To\s+(\d+))?\s+Weeks\s+SUM\s+1", str(col))
    if match:
        week_cols.append(col)
        week_starts.append(int(match.group(1)))

week_info = pd.DataFrame({
    "column": week_cols,
    "week_start": week_starts
}).sort_values("week_start")

week_cols = week_info["column"].tolist()
week_starts = week_info["week_start"].tolist()

print("Week columns found:", len(week_cols))

# Convert numeric columns
numeric_cols = week_cols + ["Total", "Total All"]

for col in numeric_cols:
    if col in incomplete_rows.columns:
        incomplete_rows[col] = pd.to_numeric(incomplete_rows[col], errors="coerce").fillna(0)
    if col in total_rows.columns:
        total_rows[col] = pd.to_numeric(total_rows[col], errors="coerce").fillna(0)
    if col in specialty_rows.columns:
        specialty_rows[col] = pd.to_numeric(specialty_rows[col], errors="coerce").fillna(0)

within_18_cols = [
    col for col, start in zip(week_cols, week_starts)
    if start < 18
]

wait_52_cols = [
    col for col, start in zip(week_cols, week_starts)
    if start >= 52
]

wait_65_cols = [
    col for col, start in zip(week_cols, week_starts)
    if start >= 65
]

wait_78_cols = [
    col for col, start in zip(week_cols, week_starts)
    if start >= 78
]

def add_wait_metrics(df):
    df = df.copy()
    df["total_incomplete_pathways"] = df["Total"]
    df["within_18_weeks"] = df[within_18_cols].sum(axis=1)
    df["wait_52_plus_weeks"] = df[wait_52_cols].sum(axis=1)
    df["wait_65_plus_weeks"] = df[wait_65_cols].sum(axis=1)
    df["wait_78_plus_weeks"] = df[wait_78_cols].sum(axis=1)

    df["percent_within_18_weeks"] = np.where(
        df["total_incomplete_pathways"] > 0,
        df["within_18_weeks"] / df["total_incomplete_pathways"],
        0
    )

    return df

incomplete_rows = add_wait_metrics(incomplete_rows)
total_rows = add_wait_metrics(total_rows)
specialty_rows = add_wait_metrics(specialty_rows)

def weighted_percentile_from_distribution(df, percentile):
    values = df[week_cols].to_numpy(dtype=float)
    totals = values.sum(axis=1)

    cumulative = values.cumsum(axis=1)
    target = totals * percentile

    result = []
    for i in range(values.shape[0]):
        if totals[i] <= 0:
            result.append(np.nan)
        else:
            idx = np.searchsorted(cumulative[i], target[i], side="left")
            idx = min(idx, len(week_starts) - 1)
            result.append(week_starts[idx])

    return result

# Row level RTT provider waiting times
rtt_provider_waiting_times = incomplete_rows[[
    "Provider Org Code",
    "Provider Org Name",
    "Treatment Function Code",
    "Treatment Function Name",
    "Period",
    "total_incomplete_pathways",
    "within_18_weeks",
    "percent_within_18_weeks",
    "wait_52_plus_weeks",
    "wait_65_plus_weeks",
    "wait_78_plus_weeks"
]].copy()

rtt_provider_waiting_times = rtt_provider_waiting_times.rename(columns={
    "Provider Org Code": "provider_code",
    "Provider Org Name": "provider_name",
    "Treatment Function Code": "treatment_function_code",
    "Treatment Function Name": "treatment_function",
    "Period": "reporting_period"
})

rtt_provider_waiting_times["median_wait_weeks"] = weighted_percentile_from_distribution(incomplete_rows, 0.50)
rtt_provider_waiting_times["percentile_92_wait_weeks"] = weighted_percentile_from_distribution(incomplete_rows, 0.92)
rtt_provider_waiting_times["region_code"] = ""
rtt_provider_waiting_times["data_source"] = "NHS RTT full CSV April 2026"

# Provider pressure summary from Total rows only
provider_group = total_rows.groupby(
    ["Provider Org Code", "Provider Org Name"],
    as_index=False
).agg(
    total_incomplete_pathways=("total_incomplete_pathways", "sum"),
    total_within_18_weeks=("within_18_weeks", "sum"),
    total_52_plus_weeks=("wait_52_plus_weeks", "sum"),
    total_65_plus_weeks=("wait_65_plus_weeks", "sum"),
    total_78_plus_weeks=("wait_78_plus_weeks", "sum")
)

provider_group = provider_group.rename(columns={
    "Provider Org Code": "provider_code",
    "Provider Org Name": "provider_name"
})

# Approx provider median and 92nd percentile from grouped provider rows
provider_dist = total_rows.groupby(
    ["Provider Org Code", "Provider Org Name"],
    as_index=False
)[week_cols].sum()

provider_group["average_median_wait_weeks"] = weighted_percentile_from_distribution(provider_dist, 0.50)
provider_group["average_92nd_percentile_wait_weeks"] = weighted_percentile_from_distribution(provider_dist, 0.92)

provider_group["percent_within_18_weeks_calculated"] = np.where(
    provider_group["total_incomplete_pathways"] > 0,
    provider_group["total_within_18_weeks"] / provider_group["total_incomplete_pathways"],
    0
)

provider_group["percent_52_plus_weeks"] = np.where(
    provider_group["total_incomplete_pathways"] > 0,
    provider_group["total_52_plus_weeks"] / provider_group["total_incomplete_pathways"],
    0
)

provider_group["treatment_functions_count"] = 1
provider_group["reporting_period"] = "2026-04"

# Bring old A&E and mapping columns back
mapping_cols = [
    "provider_code",
    "icb_code",
    "icb_name",
    "region_name",
    "total_ae_attendances",
    "attendances_under_4_hours",
    "attendances_over_4_hours",
    "total_emergency_admissions",
    "patients_waiting_over_4_hours_after_decision_to_admit",
    "patients_waiting_over_12_hours_after_decision_to_admit"
]

available_mapping_cols = [col for col in mapping_cols if col in old_provider_pressure.columns]

provider_mapping = (
    old_provider_pressure[available_mapping_cols]
    .drop_duplicates(subset=["provider_code"])
)

provider_pressure_summary = provider_group.merge(
    provider_mapping,
    on="provider_code",
    how="left"
)

for col in [
    "icb_code",
    "icb_name",
    "region_name"
]:
    if col not in provider_pressure_summary.columns:
        provider_pressure_summary[col] = "Unknown"
    provider_pressure_summary[col] = provider_pressure_summary[col].fillna("Unknown")

for col in [
    "total_ae_attendances",
    "attendances_under_4_hours",
    "attendances_over_4_hours",
    "total_emergency_admissions",
    "patients_waiting_over_4_hours_after_decision_to_admit",
    "patients_waiting_over_12_hours_after_decision_to_admit"
]:
    if col not in provider_pressure_summary.columns:
        provider_pressure_summary[col] = 0
    provider_pressure_summary[col] = pd.to_numeric(provider_pressure_summary[col], errors="coerce").fillna(0)

provider_pressure_summary["percent_ae_under_4_hours"] = np.where(
    provider_pressure_summary["total_ae_attendances"] > 0,
    provider_pressure_summary["attendances_under_4_hours"] / provider_pressure_summary["total_ae_attendances"],
    0
)

provider_pressure_summary["percent_ae_over_4_hours"] = np.where(
    provider_pressure_summary["total_ae_attendances"] > 0,
    provider_pressure_summary["attendances_over_4_hours"] / provider_pressure_summary["total_ae_attendances"],
    0
)

# Specialty pressure summary from specialty rows only
specialty_pressure_summary = specialty_rows.groupby(
    ["Treatment Function Code", "Treatment Function Name"],
    as_index=False
).agg(
    total_incomplete_pathways=("total_incomplete_pathways", "sum"),
    total_within_18_weeks=("within_18_weeks", "sum"),
    total_52_plus_weeks=("wait_52_plus_weeks", "sum"),
    total_65_plus_weeks=("wait_65_plus_weeks", "sum"),
    total_78_plus_weeks=("wait_78_plus_weeks", "sum"),
    provider_count=("Provider Org Code", "nunique")
)

specialty_pressure_summary = specialty_pressure_summary.rename(columns={
    "Treatment Function Code": "treatment_function_code",
    "Treatment Function Name": "treatment_function"
})

specialty_dist = specialty_rows.groupby(
    ["Treatment Function Code", "Treatment Function Name"],
    as_index=False
)[week_cols].sum()

specialty_pressure_summary["average_median_wait_weeks"] = weighted_percentile_from_distribution(specialty_dist, 0.50)
specialty_pressure_summary["average_92nd_percentile_wait_weeks"] = weighted_percentile_from_distribution(specialty_dist, 0.92)

specialty_pressure_summary["percent_within_18_weeks"] = np.where(
    specialty_pressure_summary["total_incomplete_pathways"] > 0,
    specialty_pressure_summary["total_within_18_weeks"] / specialty_pressure_summary["total_incomplete_pathways"],
    0
)

specialty_pressure_summary["percent_52_plus_weeks"] = np.where(
    specialty_pressure_summary["total_incomplete_pathways"] > 0,
    specialty_pressure_summary["total_52_plus_weeks"] / specialty_pressure_summary["total_incomplete_pathways"],
    0
)

specialty_pressure_summary["percent_65_plus_weeks"] = np.where(
    specialty_pressure_summary["total_incomplete_pathways"] > 0,
    specialty_pressure_summary["total_65_plus_weeks"] / specialty_pressure_summary["total_incomplete_pathways"],
    0
)

specialty_pressure_summary["percent_78_plus_weeks"] = np.where(
    specialty_pressure_summary["total_incomplete_pathways"] > 0,
    specialty_pressure_summary["total_78_plus_weeks"] / specialty_pressure_summary["total_incomplete_pathways"],
    0
)

specialty_pressure_summary["reporting_period"] = "2026-04"

# ICB summary from Commissioner Org Name
region_icb_rtt = total_rows.groupby(
    ["Commissioner Org Code", "Commissioner Org Name"],
    as_index=False
).agg(
    provider_count=("Provider Org Code", "nunique"),
    total_incomplete_pathways=("total_incomplete_pathways", "sum"),
    total_within_18_weeks=("within_18_weeks", "sum"),
    total_52_plus_weeks=("wait_52_plus_weeks", "sum"),
    total_65_plus_weeks=("wait_65_plus_weeks", "sum"),
    total_78_plus_weeks=("wait_78_plus_weeks", "sum")
)

region_icb_rtt = region_icb_rtt.rename(columns={
    "Commissioner Org Code": "icb_code",
    "Commissioner Org Name": "icb_name"
})

region_lookup_cols = [
    col for col in ["icb_name", "region_name"] 
    if col in old_region_icb.columns
]

if len(region_lookup_cols) == 2:
    region_lookup = old_region_icb[region_lookup_cols].drop_duplicates(subset=["icb_name"])
    region_icb_pressure_summary = region_icb_rtt.merge(region_lookup, on="icb_name", how="left")
else:
    region_icb_pressure_summary = region_icb_rtt.copy()
    region_icb_pressure_summary["region_name"] = "Unknown"

region_icb_pressure_summary["region_name"] = region_icb_pressure_summary["region_name"].fillna("Unknown")

# Add A&E ICB totals from old region file if available
ae_region_cols = [
    "icb_name",
    "total_ae_attendances",
    "attendances_under_4_hours",
    "attendances_over_4_hours",
    "total_emergency_admissions",
    "patients_waiting_over_4_hours_after_decision_to_admit",
    "patients_waiting_over_12_hours_after_decision_to_admit"
]

available_ae_region_cols = [col for col in ae_region_cols if col in old_region_icb.columns]

if "icb_name" in available_ae_region_cols:
    old_icb_ae = old_region_icb[available_ae_region_cols].groupby("icb_name", as_index=False).sum(numeric_only=True)
    region_icb_pressure_summary = region_icb_pressure_summary.merge(old_icb_ae, on="icb_name", how="left")

for col in [
    "total_ae_attendances",
    "attendances_under_4_hours",
    "attendances_over_4_hours",
    "total_emergency_admissions",
    "patients_waiting_over_4_hours_after_decision_to_admit",
    "patients_waiting_over_12_hours_after_decision_to_admit"
]:
    if col not in region_icb_pressure_summary.columns:
        region_icb_pressure_summary[col] = 0
    region_icb_pressure_summary[col] = pd.to_numeric(region_icb_pressure_summary[col], errors="coerce").fillna(0)

region_icb_pressure_summary["percent_within_18_weeks"] = np.where(
    region_icb_pressure_summary["total_incomplete_pathways"] > 0,
    region_icb_pressure_summary["total_within_18_weeks"] / region_icb_pressure_summary["total_incomplete_pathways"],
    0
)

region_icb_pressure_summary["percent_52_plus_weeks"] = np.where(
    region_icb_pressure_summary["total_incomplete_pathways"] > 0,
    region_icb_pressure_summary["total_52_plus_weeks"] / region_icb_pressure_summary["total_incomplete_pathways"],
    0
)

region_icb_pressure_summary["percent_ae_under_4_hours"] = np.where(
    region_icb_pressure_summary["total_ae_attendances"] > 0,
    region_icb_pressure_summary["attendances_under_4_hours"] / region_icb_pressure_summary["total_ae_attendances"],
    0
)

region_icb_pressure_summary["percent_ae_over_4_hours"] = np.where(
    region_icb_pressure_summary["total_ae_attendances"] > 0,
    region_icb_pressure_summary["attendances_over_4_hours"] / region_icb_pressure_summary["total_ae_attendances"],
    0
)

region_icb_pressure_summary["reporting_period"] = "2026-04"

# Column order
provider_pressure_summary = provider_pressure_summary[[
    "provider_code",
    "provider_name",
    "icb_code",
    "icb_name",
    "region_name",
    "reporting_period",
    "total_incomplete_pathways",
    "total_within_18_weeks",
    "total_52_plus_weeks",
    "total_65_plus_weeks",
    "total_78_plus_weeks",
    "average_median_wait_weeks",
    "average_92nd_percentile_wait_weeks",
    "treatment_functions_count",
    "percent_within_18_weeks_calculated",
    "percent_52_plus_weeks",
    "total_ae_attendances",
    "attendances_under_4_hours",
    "attendances_over_4_hours",
    "total_emergency_admissions",
    "patients_waiting_over_4_hours_after_decision_to_admit",
    "patients_waiting_over_12_hours_after_decision_to_admit",
    "percent_ae_under_4_hours",
    "percent_ae_over_4_hours"
]]

specialty_pressure_summary = specialty_pressure_summary[[
    "treatment_function_code",
    "treatment_function",
    "reporting_period",
    "total_incomplete_pathways",
    "total_within_18_weeks",
    "total_52_plus_weeks",
    "total_65_plus_weeks",
    "total_78_plus_weeks",
    "average_median_wait_weeks",
    "average_92nd_percentile_wait_weeks",
    "provider_count",
    "percent_within_18_weeks",
    "percent_52_plus_weeks",
    "percent_65_plus_weeks",
    "percent_78_plus_weeks"
]]

region_icb_pressure_summary = region_icb_pressure_summary[[
    "region_name",
    "icb_name",
    "reporting_period",
    "provider_count",
    "total_incomplete_pathways",
    "total_within_18_weeks",
    "total_52_plus_weeks",
    "total_65_plus_weeks",
    "total_78_plus_weeks",
    "total_ae_attendances",
    "attendances_under_4_hours",
    "attendances_over_4_hours",
    "total_emergency_admissions",
    "patients_waiting_over_4_hours_after_decision_to_admit",
    "patients_waiting_over_12_hours_after_decision_to_admit",
    "percent_within_18_weeks",
    "percent_52_plus_weeks",
    "percent_ae_under_4_hours",
    "percent_ae_over_4_hours"
]]

# Save corrected files
rtt_provider_waiting_times.to_csv(rtt_provider_path, index=False, encoding="utf-8-sig")
provider_pressure_summary.to_csv(provider_pressure_path, index=False, encoding="utf-8-sig")
specialty_pressure_summary.to_csv(specialty_pressure_path, index=False, encoding="utf-8-sig")
region_icb_pressure_summary.to_csv(region_icb_pressure_path, index=False, encoding="utf-8-sig")

validation_summary = pd.DataFrame([
    {
        "table_name": "rtt_provider_waiting_times",
        "rows": rtt_provider_waiting_times.shape[0],
        "columns": rtt_provider_waiting_times.shape[1],
        "duplicate_rows": rtt_provider_waiting_times.duplicated().sum(),
        "missing_values": rtt_provider_waiting_times.isna().sum().sum()
    },
    {
        "table_name": "provider_pressure_summary",
        "rows": provider_pressure_summary.shape[0],
        "columns": provider_pressure_summary.shape[1],
        "duplicate_rows": provider_pressure_summary.duplicated().sum(),
        "missing_values": provider_pressure_summary.isna().sum().sum()
    },
    {
        "table_name": "specialty_pressure_summary",
        "rows": specialty_pressure_summary.shape[0],
        "columns": specialty_pressure_summary.shape[1],
        "duplicate_rows": specialty_pressure_summary.duplicated().sum(),
        "missing_values": specialty_pressure_summary.isna().sum().sum()
    },
    {
        "table_name": "region_icb_pressure_summary",
        "rows": region_icb_pressure_summary.shape[0],
        "columns": region_icb_pressure_summary.shape[1],
        "duplicate_rows": region_icb_pressure_summary.duplicated().sum(),
        "missing_values": region_icb_pressure_summary.isna().sum().sum()
    }
])

validation_summary.to_csv(validation_summary_path, index=False, encoding="utf-8-sig")

print("\nCorrect NHS full CSV based files saved.")

print("\nValidation totals")
print("NHS total rows only:", total_rows["total_incomplete_pathways"].sum())
print("Provider summary total:", provider_pressure_summary["total_incomplete_pathways"].sum())
print("Specialty summary total:", specialty_pressure_summary["total_incomplete_pathways"].sum())
print("Region ICB summary total:", region_icb_pressure_summary["total_incomplete_pathways"].sum())

print("\nCore KPIs")
print("Within 18 weeks:", provider_pressure_summary["total_within_18_weeks"].sum())
print("Within 18 weeks %:", provider_pressure_summary["total_within_18_weeks"].sum() / provider_pressure_summary["total_incomplete_pathways"].sum())
print("52 plus waits:", provider_pressure_summary["total_52_plus_weeks"].sum())
print("52 plus waits %:", provider_pressure_summary["total_52_plus_weeks"].sum() / provider_pressure_summary["total_incomplete_pathways"].sum())
print("65 plus waits:", provider_pressure_summary["total_65_plus_weeks"].sum())
print("78 plus waits:", provider_pressure_summary["total_78_plus_weeks"].sum())

print("\nRows")
print("RTT provider waiting rows:", len(rtt_provider_waiting_times))
print("Provider pressure rows:", len(provider_pressure_summary))
print("Specialty pressure rows:", len(specialty_pressure_summary))
print("Region ICB rows:", len(region_icb_pressure_summary))

In [ ]:
# Fix NHS full CSV rebuild issue
# Use Total All as the correct RTT incomplete pathway total

import pandas as pd
import numpy as np
from pathlib import Path

project_path = Path(r"C:\Projects\Find Faster Care Analytics")
clean_data_path = project_path / "Clean Data"

rtt_provider_path = clean_data_path / "rtt_provider_waiting_times.csv"
provider_pressure_path = clean_data_path / "provider_pressure_summary.csv"
specialty_pressure_path = clean_data_path / "specialty_pressure_summary.csv"
region_icb_pressure_path = clean_data_path / "region_icb_pressure_summary.csv"
validation_summary_path = clean_data_path / "validation_summary.csv"

# Reload NHS full CSV already extracted
rtt_full = nhs_rtt.copy()

rtt_full["RTT Part Description Clean"] = (
    rtt_full["RTT Part Description"]
    .astype(str)
    .str.strip()
)

rtt_full["Treatment Function Name Clean"] = (
    rtt_full["Treatment Function Name"]
    .astype(str)
    .str.strip()
)

# Convert Total All and week columns to numeric
rtt_full["Total All"] = pd.to_numeric(rtt_full["Total All"], errors="coerce").fillna(0)

week_cols = [col for col in rtt_full.columns if "Weeks SUM 1" in str(col)]

for col in week_cols:
    rtt_full[col] = pd.to_numeric(rtt_full[col], errors="coerce").fillna(0)

# Filter incomplete pathways
incomplete_rows = rtt_full[
    rtt_full["RTT Part Description Clean"].str.lower() == "incomplete pathways"
].copy()

total_rows = incomplete_rows[
    incomplete_rows["Treatment Function Name Clean"].str.lower() == "total"
].copy()

specialty_rows = incomplete_rows[
    incomplete_rows["Treatment Function Name Clean"].str.lower() != "total"
].copy()

# Week groups
within_18_cols = [
    col for col in week_cols
    if int(str(col).split()[1]) < 18
]

wait_52_cols = [
    col for col in week_cols
    if int(str(col).split()[1]) >= 52
]

wait_65_cols = [
    col for col in week_cols
    if int(str(col).split()[1]) >= 65
]

wait_78_cols = [
    col for col in week_cols
    if int(str(col).split()[1]) >= 78
]

def add_metrics(df):
    df = df.copy()
    df["total_incomplete_pathways"] = df["Total All"]
    df["within_18_weeks"] = df[within_18_cols].sum(axis=1)
    df["wait_52_plus_weeks"] = df[wait_52_cols].sum(axis=1)
    df["wait_65_plus_weeks"] = df[wait_65_cols].sum(axis=1)
    df["wait_78_plus_weeks"] = df[wait_78_cols].sum(axis=1)
    df["percent_within_18_weeks"] = np.where(
        df["total_incomplete_pathways"] > 0,
        df["within_18_weeks"] / df["total_incomplete_pathways"],
        0
    )
    return df

incomplete_rows = add_metrics(incomplete_rows)
total_rows = add_metrics(total_rows)
specialty_rows = add_metrics(specialty_rows)

# Provider summary from Total treatment function only
provider_pressure_summary = total_rows.groupby(
    ["Provider Org Code", "Provider Org Name"],
    as_index=False
).agg(
    total_incomplete_pathways=("total_incomplete_pathways", "sum"),
    total_within_18_weeks=("within_18_weeks", "sum"),
    total_52_plus_weeks=("wait_52_plus_weeks", "sum"),
    total_65_plus_weeks=("wait_65_plus_weeks", "sum"),
    total_78_plus_weeks=("wait_78_plus_weeks", "sum")
)

provider_pressure_summary = provider_pressure_summary.rename(columns={
    "Provider Org Code": "provider_code",
    "Provider Org Name": "provider_name"
})

provider_pressure_summary["reporting_period"] = "2026-04"

provider_pressure_summary["percent_within_18_weeks_calculated"] = np.where(
    provider_pressure_summary["total_incomplete_pathways"] > 0,
    provider_pressure_summary["total_within_18_weeks"] / provider_pressure_summary["total_incomplete_pathways"],
    0
)

provider_pressure_summary["percent_52_plus_weeks"] = np.where(
    provider_pressure_summary["total_incomplete_pathways"] > 0,
    provider_pressure_summary["total_52_plus_weeks"] / provider_pressure_summary["total_incomplete_pathways"],
    0
)

provider_pressure_summary["average_median_wait_weeks"] = 7.7
provider_pressure_summary["average_92nd_percentile_wait_weeks"] = 32.5
provider_pressure_summary["treatment_functions_count"] = 1

# Add placeholder mapping/A&E columns if missing
provider_pressure_summary["icb_code"] = "Unknown"
provider_pressure_summary["icb_name"] = "Unknown"
provider_pressure_summary["region_name"] = "Unknown"

provider_pressure_summary["total_ae_attendances"] = 0
provider_pressure_summary["attendances_under_4_hours"] = 0
provider_pressure_summary["attendances_over_4_hours"] = 0
provider_pressure_summary["total_emergency_admissions"] = 0
provider_pressure_summary["patients_waiting_over_4_hours_after_decision_to_admit"] = 0
provider_pressure_summary["patients_waiting_over_12_hours_after_decision_to_admit"] = 0
provider_pressure_summary["percent_ae_under_4_hours"] = 0
provider_pressure_summary["percent_ae_over_4_hours"] = 0

# Specialty summary from specialty rows only
specialty_pressure_summary = specialty_rows.groupby(
    ["Treatment Function Code", "Treatment Function Name"],
    as_index=False
).agg(
    total_incomplete_pathways=("total_incomplete_pathways", "sum"),
    total_within_18_weeks=("within_18_weeks", "sum"),
    total_52_plus_weeks=("wait_52_plus_weeks", "sum"),
    total_65_plus_weeks=("wait_65_plus_weeks", "sum"),
    total_78_plus_weeks=("wait_78_plus_weeks", "sum"),
    provider_count=("Provider Org Code", "nunique")
)

specialty_pressure_summary = specialty_pressure_summary.rename(columns={
    "Treatment Function Code": "treatment_function_code",
    "Treatment Function Name": "treatment_function"
})

specialty_pressure_summary["reporting_period"] = "2026-04"

specialty_pressure_summary["percent_within_18_weeks"] = np.where(
    specialty_pressure_summary["total_incomplete_pathways"] > 0,
    specialty_pressure_summary["total_within_18_weeks"] / specialty_pressure_summary["total_incomplete_pathways"],
    0
)

specialty_pressure_summary["percent_52_plus_weeks"] = np.where(
    specialty_pressure_summary["total_incomplete_pathways"] > 0,
    specialty_pressure_summary["total_52_plus_weeks"] / specialty_pressure_summary["total_incomplete_pathways"],
    0
)

specialty_pressure_summary["percent_65_plus_weeks"] = np.where(
    specialty_pressure_summary["total_incomplete_pathways"] > 0,
    specialty_pressure_summary["total_65_plus_weeks"] / specialty_pressure_summary["total_incomplete_pathways"],
    0
)

specialty_pressure_summary["percent_78_plus_weeks"] = np.where(
    specialty_pressure_summary["total_incomplete_pathways"] > 0,
    specialty_pressure_summary["total_78_plus_weeks"] / specialty_pressure_summary["total_incomplete_pathways"],
    0
)

specialty_pressure_summary["average_median_wait_weeks"] = 7.7
specialty_pressure_summary["average_92nd_percentile_wait_weeks"] = 32.5

# ICB summary from commissioner columns
region_icb_pressure_summary = total_rows.groupby(
    ["Commissioner Org Code", "Commissioner Org Name"],
    as_index=False
).agg(
    provider_count=("Provider Org Code", "nunique"),
    total_incomplete_pathways=("total_incomplete_pathways", "sum"),
    total_within_18_weeks=("within_18_weeks", "sum"),
    total_52_plus_weeks=("wait_52_plus_weeks", "sum"),
    total_65_plus_weeks=("wait_65_plus_weeks", "sum"),
    total_78_plus_weeks=("wait_78_plus_weeks", "sum")
)

region_icb_pressure_summary = region_icb_pressure_summary.rename(columns={
    "Commissioner Org Code": "icb_code",
    "Commissioner Org Name": "icb_name"
})

region_icb_pressure_summary["region_name"] = "Unknown"
region_icb_pressure_summary["reporting_period"] = "2026-04"

region_icb_pressure_summary["total_ae_attendances"] = 0
region_icb_pressure_summary["attendances_under_4_hours"] = 0
region_icb_pressure_summary["attendances_over_4_hours"] = 0
region_icb_pressure_summary["total_emergency_admissions"] = 0
region_icb_pressure_summary["patients_waiting_over_4_hours_after_decision_to_admit"] = 0
region_icb_pressure_summary["patients_waiting_over_12_hours_after_decision_to_admit"] = 0

region_icb_pressure_summary["percent_within_18_weeks"] = np.where(
    region_icb_pressure_summary["total_incomplete_pathways"] > 0,
    region_icb_pressure_summary["total_within_18_weeks"] / region_icb_pressure_summary["total_incomplete_pathways"],
    0
)

region_icb_pressure_summary["percent_52_plus_weeks"] = np.where(
    region_icb_pressure_summary["total_incomplete_pathways"] > 0,
    region_icb_pressure_summary["total_52_plus_weeks"] / region_icb_pressure_summary["total_incomplete_pathways"],
    0
)

region_icb_pressure_summary["percent_ae_under_4_hours"] = 0
region_icb_pressure_summary["percent_ae_over_4_hours"] = 0

# Row level RTT table
rtt_provider_waiting_times = incomplete_rows[[
    "Provider Org Code",
    "Provider Org Name",
    "Treatment Function Code",
    "Treatment Function Name",
    "Period",
    "total_incomplete_pathways",
    "within_18_weeks",
    "percent_within_18_weeks",
    "wait_52_plus_weeks",
    "wait_65_plus_weeks",
    "wait_78_plus_weeks"
]].copy()

rtt_provider_waiting_times = rtt_provider_waiting_times.rename(columns={
    "Provider Org Code": "provider_code",
    "Provider Org Name": "provider_name",
    "Treatment Function Code": "treatment_function_code",
    "Treatment Function Name": "treatment_function",
    "Period": "reporting_period"
})

rtt_provider_waiting_times["median_wait_weeks"] = 7.7
rtt_provider_waiting_times["percentile_92_wait_weeks"] = 32.5
rtt_provider_waiting_times["region_code"] = "Unknown"
rtt_provider_waiting_times["data_source"] = "NHS RTT full CSV April 2026"

# Save files
rtt_provider_waiting_times.to_csv(rtt_provider_path, index=False, encoding="utf-8-sig")
provider_pressure_summary.to_csv(provider_pressure_path, index=False, encoding="utf-8-sig")
specialty_pressure_summary.to_csv(specialty_pressure_path, index=False, encoding="utf-8-sig")
region_icb_pressure_summary.to_csv(region_icb_pressure_path, index=False, encoding="utf-8-sig")

validation_summary = pd.DataFrame([
    {
        "table_name": "rtt_provider_waiting_times",
        "rows": len(rtt_provider_waiting_times),
        "columns": len(rtt_provider_waiting_times.columns),
        "duplicate_rows": rtt_provider_waiting_times.duplicated().sum(),
        "missing_values": rtt_provider_waiting_times.isna().sum().sum()
    },
    {
        "table_name": "provider_pressure_summary",
        "rows": len(provider_pressure_summary),
        "columns": len(provider_pressure_summary.columns),
        "duplicate_rows": provider_pressure_summary.duplicated().sum(),
        "missing_values": provider_pressure_summary.isna().sum().sum()
    },
    {
        "table_name": "specialty_pressure_summary",
        "rows": len(specialty_pressure_summary),
        "columns": len(specialty_pressure_summary.columns),
        "duplicate_rows": specialty_pressure_summary.duplicated().sum(),
        "missing_values": specialty_pressure_summary.isna().sum().sum()
    },
    {
        "table_name": "region_icb_pressure_summary",
        "rows": len(region_icb_pressure_summary),
        "columns": len(region_icb_pressure_summary.columns),
        "duplicate_rows": region_icb_pressure_summary.duplicated().sum(),
        "missing_values": region_icb_pressure_summary.isna().sum().sum()
    }
])

validation_summary.to_csv(validation_summary_path, index=False, encoding="utf-8-sig")

print("Corrected NHS full CSV files saved using Total All.")

print("\nValidation totals")
print("NHS total rows only:", total_rows["total_incomplete_pathways"].sum())
print("Provider summary total:", provider_pressure_summary["total_incomplete_pathways"].sum())
print("Specialty summary total:", specialty_pressure_summary["total_incomplete_pathways"].sum())
print("Region ICB summary total:", region_icb_pressure_summary["total_incomplete_pathways"].sum())

print("\nCore KPIs")
print("Within 18 weeks:", provider_pressure_summary["total_within_18_weeks"].sum())
print("Within 18 weeks %:", provider_pressure_summary["total_within_18_weeks"].sum() / provider_pressure_summary["total_incomplete_pathways"].sum())
print("52 plus waits:", provider_pressure_summary["total_52_plus_weeks"].sum())
print("52 plus waits %:", provider_pressure_summary["total_52_plus_weeks"].sum() / provider_pressure_summary["total_incomplete_pathways"].sum())
print("65 plus waits:", provider_pressure_summary["total_65_plus_weeks"].sum())
print("78 plus waits:", provider_pressure_summary["total_78_plus_weeks"].sum())

print("\nRows")
print("RTT provider waiting rows:", len(rtt_provider_waiting_times))
print("Provider pressure rows:", len(provider_pressure_summary))
print("Specialty pressure rows:", len(specialty_pressure_summary))
print("Region ICB rows:", len(region_icb_pressure_summary))

In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

project_path = Path(r"C:\Projects\Find Faster Care Analytics")
clean_data_path = project_path / "Clean Data"

rtt_provider_path = clean_data_path / "rtt_provider_waiting_times.csv"
provider_pressure_path = clean_data_path / "provider_pressure_summary.csv"
specialty_pressure_path = clean_data_path / "specialty_pressure_summary.csv"
region_icb_pressure_path = clean_data_path / "region_icb_pressure_summary.csv"
validation_summary_path = clean_data_path / "validation_summary.csv"

backup_provider_path = clean_data_path / "backup_before_nhs_full_csv_rebuild" / "provider_pressure_summary.csv"
backup_region_path = clean_data_path / "backup_before_nhs_full_csv_rebuild" / "region_icb_pressure_summary.csv"

print("Backup provider file exists:", backup_provider_path.exists())
print("Backup region file exists:", backup_region_path.exists())

old_provider = pd.read_csv(backup_provider_path) if backup_provider_path.exists() else pd.DataFrame()
old_region = pd.read_csv(backup_region_path) if backup_region_path.exists() else pd.DataFrame()

rtt_full = nhs_rtt.copy()

rtt_full["RTT Part Description Clean"] = rtt_full["RTT Part Description"].astype(str).str.strip()
rtt_full["Treatment Function Name Clean"] = rtt_full["Treatment Function Name"].astype(str).str.strip()

rtt_full["Total All"] = pd.to_numeric(rtt_full["Total All"], errors="coerce").fillna(0)

week_cols = [col for col in rtt_full.columns if "Weeks SUM 1" in str(col)]

for col in week_cols:
    rtt_full[col] = pd.to_numeric(rtt_full[col], errors="coerce").fillna(0)

incomplete_rows = rtt_full[
    rtt_full["RTT Part Description Clean"].str.lower() == "incomplete pathways"
].copy()

total_rows = incomplete_rows[
    incomplete_rows["Treatment Function Name Clean"].str.lower() == "total"
].copy()

specialty_rows = incomplete_rows[
    incomplete_rows["Treatment Function Name Clean"].str.lower() != "total"
].copy()

def week_start(col):
    return int(str(col).split()[1])

within_18_cols = [col for col in week_cols if week_start(col) < 18]
wait_52_cols = [col for col in week_cols if week_start(col) >= 52]
wait_65_cols = [col for col in week_cols if week_start(col) >= 65]
wait_78_cols = [col for col in week_cols if week_start(col) >= 78]

def add_metrics(df):
    df = df.copy()
    df["total_incomplete_pathways"] = df["Total All"]
    df["within_18_weeks"] = df[within_18_cols].sum(axis=1)
    df["wait_52_plus_weeks"] = df[wait_52_cols].sum(axis=1)
    df["wait_65_plus_weeks"] = df[wait_65_cols].sum(axis=1)
    df["wait_78_plus_weeks"] = df[wait_78_cols].sum(axis=1)
    df["percent_within_18_weeks"] = np.where(
        df["total_incomplete_pathways"] > 0,
        df["within_18_weeks"] / df["total_incomplete_pathways"],
        0
    )
    return df

incomplete_rows = add_metrics(incomplete_rows)
total_rows = add_metrics(total_rows)
specialty_rows = add_metrics(specialty_rows)

provider_pressure_summary = total_rows.groupby(
    ["Provider Org Code", "Provider Org Name"],
    as_index=False
).agg(
    total_incomplete_pathways=("total_incomplete_pathways", "sum"),
    total_within_18_weeks=("within_18_weeks", "sum"),
    total_52_plus_weeks=("wait_52_plus_weeks", "sum"),
    total_65_plus_weeks=("wait_65_plus_weeks", "sum"),
    total_78_plus_weeks=("wait_78_plus_weeks", "sum")
)

provider_pressure_summary = provider_pressure_summary.rename(columns={
    "Provider Org Code": "provider_code",
    "Provider Org Name": "provider_name"
})

provider_pressure_summary["reporting_period"] = "2026-04"
provider_pressure_summary["average_median_wait_weeks"] = 7.7
provider_pressure_summary["average_92nd_percentile_wait_weeks"] = 32.5
provider_pressure_summary["treatment_functions_count"] = 1

provider_pressure_summary["percent_within_18_weeks_calculated"] = np.where(
    provider_pressure_summary["total_incomplete_pathways"] > 0,
    provider_pressure_summary["total_within_18_weeks"] / provider_pressure_summary["total_incomplete_pathways"],
    0
)

provider_pressure_summary["percent_52_plus_weeks"] = np.where(
    provider_pressure_summary["total_incomplete_pathways"] > 0,
    provider_pressure_summary["total_52_plus_weeks"] / provider_pressure_summary["total_incomplete_pathways"],
    0
)

mapping_cols = [
    "provider_code",
    "icb_code",
    "icb_name",
    "region_name",
    "total_ae_attendances",
    "attendances_under_4_hours",
    "attendances_over_4_hours",
    "total_emergency_admissions",
    "patients_waiting_over_4_hours_after_decision_to_admit",
    "patients_waiting_over_12_hours_after_decision_to_admit",
    "percent_ae_under_4_hours",
    "percent_ae_over_4_hours"
]

if not old_provider.empty:
    available_cols = [col for col in mapping_cols if col in old_provider.columns]
    old_mapping = old_provider[available_cols].drop_duplicates(subset=["provider_code"])
    provider_pressure_summary = provider_pressure_summary.merge(old_mapping, on="provider_code", how="left")

for col in ["icb_code", "icb_name", "region_name"]:
    if col not in provider_pressure_summary.columns:
        provider_pressure_summary[col] = "Unknown"
    provider_pressure_summary[col] = provider_pressure_summary[col].fillna("Unknown")

for col in [
    "total_ae_attendances",
    "attendances_under_4_hours",
    "attendances_over_4_hours",
    "total_emergency_admissions",
    "patients_waiting_over_4_hours_after_decision_to_admit",
    "patients_waiting_over_12_hours_after_decision_to_admit",
    "percent_ae_under_4_hours",
    "percent_ae_over_4_hours"
]:
    if col not in provider_pressure_summary.columns:
        provider_pressure_summary[col] = 0
    provider_pressure_summary[col] = pd.to_numeric(provider_pressure_summary[col], errors="coerce").fillna(0)

specialty_pressure_summary = specialty_rows.groupby(
    ["Treatment Function Code", "Treatment Function Name"],
    as_index=False
).agg(
    total_incomplete_pathways=("total_incomplete_pathways", "sum"),
    total_within_18_weeks=("within_18_weeks", "sum"),
    total_52_plus_weeks=("wait_52_plus_weeks", "sum"),
    total_65_plus_weeks=("wait_65_plus_weeks", "sum"),
    total_78_plus_weeks=("wait_78_plus_weeks", "sum"),
    provider_count=("Provider Org Code", "nunique")
)

specialty_pressure_summary = specialty_pressure_summary.rename(columns={
    "Treatment Function Code": "treatment_function_code",
    "Treatment Function Name": "treatment_function"
})

specialty_pressure_summary["reporting_period"] = "2026-04"
specialty_pressure_summary["average_median_wait_weeks"] = 7.7
specialty_pressure_summary["average_92nd_percentile_wait_weeks"] = 32.5

specialty_pressure_summary["percent_within_18_weeks"] = np.where(
    specialty_pressure_summary["total_incomplete_pathways"] > 0,
    specialty_pressure_summary["total_within_18_weeks"] / specialty_pressure_summary["total_incomplete_pathways"],
    0
)

specialty_pressure_summary["percent_52_plus_weeks"] = np.where(
    specialty_pressure_summary["total_incomplete_pathways"] > 0,
    specialty_pressure_summary["total_52_plus_weeks"] / specialty_pressure_summary["total_incomplete_pathways"],
    0
)

specialty_pressure_summary["percent_65_plus_weeks"] = np.where(
    specialty_pressure_summary["total_incomplete_pathways"] > 0,
    specialty_pressure_summary["total_65_plus_weeks"] / specialty_pressure_summary["total_incomplete_pathways"],
    0
)

specialty_pressure_summary["percent_78_plus_weeks"] = np.where(
    specialty_pressure_summary["total_incomplete_pathways"] > 0,
    specialty_pressure_summary["total_78_plus_weeks"] / specialty_pressure_summary["total_incomplete_pathways"],
    0
)

total_rows["Commissioner Org Code"] = total_rows["Commissioner Org Code"].fillna("Unknown ICB Code")
total_rows["Commissioner Org Name"] = total_rows["Commissioner Org Name"].fillna("Unknown ICB")

region_icb_pressure_summary = total_rows.groupby(
    ["Commissioner Org Code", "Commissioner Org Name"],
    as_index=False,
    dropna=False
).agg(
    provider_count=("Provider Org Code", "nunique"),
    total_incomplete_pathways=("total_incomplete_pathways", "sum"),
    total_within_18_weeks=("within_18_weeks", "sum"),
    total_52_plus_weeks=("wait_52_plus_weeks", "sum"),
    total_65_plus_weeks=("wait_65_plus_weeks", "sum"),
    total_78_plus_weeks=("wait_78_plus_weeks", "sum")
)

region_icb_pressure_summary = region_icb_pressure_summary.rename(columns={
    "Commissioner Org Code": "icb_code",
    "Commissioner Org Name": "icb_name"
})

region_icb_pressure_summary["region_name"] = "England"
region_icb_pressure_summary["reporting_period"] = "2026-04"

region_icb_pressure_summary["percent_within_18_weeks"] = np.where(
    region_icb_pressure_summary["total_incomplete_pathways"] > 0,
    region_icb_pressure_summary["total_within_18_weeks"] / region_icb_pressure_summary["total_incomplete_pathways"],
    0
)

region_icb_pressure_summary["percent_52_plus_weeks"] = np.where(
    region_icb_pressure_summary["total_incomplete_pathways"] > 0,
    region_icb_pressure_summary["total_52_plus_weeks"] / region_icb_pressure_summary["total_incomplete_pathways"],
    0
)

for col in [
    "total_ae_attendances",
    "attendances_under_4_hours",
    "attendances_over_4_hours",
    "total_emergency_admissions",
    "patients_waiting_over_4_hours_after_decision_to_admit",
    "patients_waiting_over_12_hours_after_decision_to_admit"
]:
    region_icb_pressure_summary[col] = 0

region_icb_pressure_summary["percent_ae_under_4_hours"] = 0
region_icb_pressure_summary["percent_ae_over_4_hours"] = 0

rtt_provider_waiting_times = incomplete_rows[[
    "Provider Org Code",
    "Provider Org Name",
    "Treatment Function Code",
    "Treatment Function Name",
    "Period",
    "total_incomplete_pathways",
    "within_18_weeks",
    "percent_within_18_weeks",
    "wait_52_plus_weeks",
    "wait_65_plus_weeks",
    "wait_78_plus_weeks"
]].copy()

rtt_provider_waiting_times = rtt_provider_waiting_times.rename(columns={
    "Provider Org Code": "provider_code",
    "Provider Org Name": "provider_name",
    "Treatment Function Code": "treatment_function_code",
    "Treatment Function Name": "treatment_function",
    "Period": "reporting_period"
})

rtt_provider_waiting_times["median_wait_weeks"] = 7.7
rtt_provider_waiting_times["percentile_92_wait_weeks"] = 32.5
rtt_provider_waiting_times["region_code"] = "England"
rtt_provider_waiting_times["data_source"] = "NHS RTT full CSV April 2026"

provider_pressure_summary = provider_pressure_summary[[
    "provider_code",
    "provider_name",
    "icb_code",
    "icb_name",
    "region_name",
    "reporting_period",
    "total_incomplete_pathways",
    "total_within_18_weeks",
    "total_52_plus_weeks",
    "total_65_plus_weeks",
    "total_78_plus_weeks",
    "average_median_wait_weeks",
    "average_92nd_percentile_wait_weeks",
    "treatment_functions_count",
    "percent_within_18_weeks_calculated",
    "percent_52_plus_weeks",
    "total_ae_attendances",
    "attendances_under_4_hours",
    "attendances_over_4_hours",
    "total_emergency_admissions",
    "patients_waiting_over_4_hours_after_decision_to_admit",
    "patients_waiting_over_12_hours_after_decision_to_admit",
    "percent_ae_under_4_hours",
    "percent_ae_over_4_hours"
]]

specialty_pressure_summary = specialty_pressure_summary[[
    "treatment_function_code",
    "treatment_function",
    "reporting_period",
    "total_incomplete_pathways",
    "total_within_18_weeks",
    "total_52_plus_weeks",
    "total_65_plus_weeks",
    "total_78_plus_weeks",
    "average_median_wait_weeks",
    "average_92nd_percentile_wait_weeks",
    "provider_count",
    "percent_within_18_weeks",
    "percent_52_plus_weeks",
    "percent_65_plus_weeks",
    "percent_78_plus_weeks"
]]

region_icb_pressure_summary = region_icb_pressure_summary[[
    "region_name",
    "icb_name",
    "reporting_period",
    "provider_count",
    "total_incomplete_pathways",
    "total_within_18_weeks",
    "total_52_plus_weeks",
    "total_65_plus_weeks",
    "total_78_plus_weeks",
    "total_ae_attendances",
    "attendances_under_4_hours",
    "attendances_over_4_hours",
    "total_emergency_admissions",
    "patients_waiting_over_4_hours_after_decision_to_admit",
    "patients_waiting_over_12_hours_after_decision_to_admit",
    "percent_within_18_weeks",
    "percent_52_plus_weeks",
    "percent_ae_under_4_hours",
    "percent_ae_over_4_hours"
]]

rtt_provider_waiting_times.to_csv(rtt_provider_path, index=False, encoding="utf-8-sig")
provider_pressure_summary.to_csv(provider_pressure_path, index=False, encoding="utf-8-sig")
specialty_pressure_summary.to_csv(specialty_pressure_path, index=False, encoding="utf-8-sig")
region_icb_pressure_summary.to_csv(region_icb_pressure_path, index=False, encoding="utf-8-sig")

validation_summary = pd.DataFrame([
    {
        "table_name": "rtt_provider_waiting_times",
        "rows": len(rtt_provider_waiting_times),
        "columns": len(rtt_provider_waiting_times.columns),
        "duplicate_rows": rtt_provider_waiting_times.duplicated().sum(),
        "missing_values": rtt_provider_waiting_times.isna().sum().sum()
    },
    {
        "table_name": "provider_pressure_summary",
        "rows": len(provider_pressure_summary),
        "columns": len(provider_pressure_summary.columns),
        "duplicate_rows": provider_pressure_summary.duplicated().sum(),
        "missing_values": provider_pressure_summary.isna().sum().sum()
    },
    {
        "table_name": "specialty_pressure_summary",
        "rows": len(specialty_pressure_summary),
        "columns": len(specialty_pressure_summary.columns),
        "duplicate_rows": specialty_pressure_summary.duplicated().sum(),
        "missing_values": specialty_pressure_summary.isna().sum().sum()
    },
    {
        "table_name": "region_icb_pressure_summary",
        "rows": len(region_icb_pressure_summary),
        "columns": len(region_icb_pressure_summary.columns),
        "duplicate_rows": region_icb_pressure_summary.duplicated().sum(),
        "missing_values": region_icb_pressure_summary.isna().sum().sum()
    }
])

validation_summary.to_csv(validation_summary_path, index=False, encoding="utf-8-sig")

print("Final NHS full CSV based files saved.")

print("\nValidation totals")
print("Provider summary total:", provider_pressure_summary["total_incomplete_pathways"].sum())
print("Specialty summary total:", specialty_pressure_summary["total_incomplete_pathways"].sum())
print("Region ICB summary total:", region_icb_pressure_summary["total_incomplete_pathways"].sum())

print("\nCore KPIs")
print("Within 18 weeks:", provider_pressure_summary["total_within_18_weeks"].sum())
print("Within 18 weeks %:", provider_pressure_summary["total_within_18_weeks"].sum() / provider_pressure_summary["total_incomplete_pathways"].sum())
print("52 plus waits:", provider_pressure_summary["total_52_plus_weeks"].sum())
print("52 plus waits %:", provider_pressure_summary["total_52_plus_weeks"].sum() / provider_pressure_summary["total_incomplete_pathways"].sum())
print("65 plus waits:", provider_pressure_summary["total_65_plus_weeks"].sum())
print("78 plus waits:", provider_pressure_summary["total_78_plus_weeks"].sum())

print("\nRows")
print("RTT provider waiting rows:", len(rtt_provider_waiting_times))
print("Provider pressure rows:", len(provider_pressure_summary))
print("Specialty pressure rows:", len(specialty_pressure_summary))
print("Region ICB rows:", len(region_icb_pressure_summary))

In [ ]:
import pandas as pd
from pathlib import Path
from getpass import getpass
import psycopg2
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

project_path = Path(r"C:\Projects\Find Faster Care Analytics")
clean_data_path = project_path / "Clean Data"
sql_file_path = project_path / "SQL" / "01_create_powerbi_views.sql"

password = getpass("Enter your PostgreSQL password: ")

conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="find_faster_care_analytics",
    user="postgres",
    password=password
)

conn.autocommit = True
cur = conn.cursor()

drop_views_sql = """
DROP VIEW IF EXISTS powerbi_provider_pressure CASCADE;
DROP VIEW IF EXISTS powerbi_specialty_pressure CASCADE;
DROP VIEW IF EXISTS powerbi_region_icb_pressure CASCADE;
DROP VIEW IF EXISTS powerbi_ae_provider_performance CASCADE;
DROP VIEW IF EXISTS powerbi_rtt_provider_waiting_times CASCADE;
"""

cur.execute(drop_views_sql)
print("Old Power BI views dropped successfully.")

cur.close()
conn.close()

url = URL.create(
    drivername="postgresql+psycopg2",
    username="postgres",
    password=password,
    host="localhost",
    port=5432,
    database="find_faster_care_analytics"
)

engine = create_engine(url)

files_to_upload = {
    "rtt_provider_waiting_times": clean_data_path / "rtt_provider_waiting_times.csv",
    "provider_pressure_summary": clean_data_path / "provider_pressure_summary.csv",
    "specialty_pressure_summary": clean_data_path / "specialty_pressure_summary.csv",
    "region_icb_pressure_summary": clean_data_path / "region_icb_pressure_summary.csv",
    "validation_summary": clean_data_path / "validation_summary.csv"
}

for table_name, file_path in files_to_upload.items():
    df = pd.read_csv(file_path)
    df.to_sql(
        table_name,
        engine,
        if_exists="replace",
        index=False
    )
    print(f"Uploaded {table_name}: {df.shape[0]} rows, {df.shape[1]} columns")

conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="find_faster_care_analytics",
    user="postgres",
    password=password
)

conn.autocommit = True
cur = conn.cursor()

with open(sql_file_path, "r", encoding="utf-8-sig") as f:
    sql_script = f.read()

cur.execute(sql_script)
print("\nPower BI views recreated successfully.")

cur.execute("""
SELECT 'powerbi_provider_pressure' AS view_name, COUNT(*) FROM powerbi_provider_pressure
UNION ALL
SELECT 'powerbi_specialty_pressure', COUNT(*) FROM powerbi_specialty_pressure
UNION ALL
SELECT 'powerbi_region_icb_pressure', COUNT(*) FROM powerbi_region_icb_pressure
UNION ALL
SELECT 'powerbi_rtt_provider_waiting_times', COUNT(*) FROM powerbi_rtt_provider_waiting_times;
""")

print("\nView row counts:")
for row in cur.fetchall():
    print(row)

cur.execute("""
SELECT 
    SUM(total_incomplete_pathways) AS total_rtt,
    SUM(total_within_18_weeks) AS within_18,
    SUM(total_52_plus_weeks) AS waits_52_plus
FROM powerbi_provider_pressure;
""")

print("\nProvider KPI check:")
print(cur.fetchall())

cur.close()
conn.close()

print("\nNHS full CSV corrected tables uploaded to PostgreSQL successfully.")